# GF-HashBA — Generative Flow-guided Hash Bundle Adjustment

Research prototype: **Hash-SDF** + **learnable cameras** + **generative prior** for unobserved regions.

## The four components

1. **Continuous Generative Score Guidance** — SDF→density proxy → flow/occupancy distillation
2. **Visibility-Gated Residual Fusion** — real multi-view `sg(V)·L_photo + (1−sg(V))·L_prior`
3. **Dual-Field + Mask Routing** — Hash-SDF FG + contracted bg NeRF; SAM masks hard-route rays
4. **Coarse-to-Fine Hash Annealing** — unlock high-res hash levels after poses lock

## Default workflow

```
§1 env → §2 config → §3 load views → §3.5 Stage-1 occupancy teacher
      → §4 modules → §5 smoke → §6.0a Stage-2 prior → §6.0 build → §6.1 train → §7 mesh
```

**Prior:** `FLOW_PRIOR_MODE="stage2"` (real MV-SAM3D Stage-2 via §6.0a).
**Visible toggle:** `FLOW_PRIOR_ON_VISIBLE=False` → Stage-2 only on blind spots; `True` → also on visible.

**Needs from `sam_baneus`:** `cameras_sphere.npz`, `baneus_scene_meta.npz`, `stage1/stage1_occupancy.npz`.
**Pinned export:** `SatelliteFuse/SamBAneus/20260811_182029/va_split/` (edit `SAM_BANEU_RUN` in §2).
**RGB/masks:** `SatelliteFuse/datasets/va_split/satellite/`.

**Flow schedule:** after Stage-1 prefit, Stage-2 leads early (basin / smoothing), then yields to photo. Adaptive cadence (~12→100). `W_FLOW` **holds 0.55 for 35%** then cosine → 0.06. Default = blind spots only; `FLOW_PRIOR_ON_VISIBLE=True` also hits visible voxels.

**Alignment:** Stage-2 densify uses the same Sim3 as prefit (`coords/64−0.5 → M → world_align → NeuS`), reuses exported Stage-1 coords — not AABB stretch / fresh SS.

**Local tests:** `python tests/test_gfhashba_alignment.py` (Sim3 / densify / mesh health).

**Debug:** after §3.5b / §6.0a / §7, paste the printed `=== DEBUG SNAPSHOT ===` JSON blocks back into chat (also saved under `OUTPUT_DIR/debug_snapshot_*.json`).

## 0 — Method overview (why each piece exists)

### The paradox this notebook solves

- **MLP NeuS** (BA-NeuS): spectral bias → smooth early geometry → good for pose BA, slow for detail.
- **Hash grids** (Instant-NGP): learn detail immediately → with wrong poses they memorize floaters and never recover.
- **Generative priors** (SDS / flow): want a statistically average shape; photometric loss wants *this* building.
- **Unbounded scenes**: sky/trees steal hash capacity from the building.

GF-HashBA's answer:

```
early iters:  only low hash levels ON  →  poses BA like an MLP
mid iters:    unlock finer levels      →  texture / edges
early:        strong flow/occ prior globally (scaffold; gate g≈0)
late:         V(x) gates photo vs prior (chisel; gate g→1)
rays:         SAM mask → fg Hash-SDF vs bg NeRF (contracted)
every K iters: SDF→σ grid → flow prior → ∇ into hash (no BP through prior)
```


## 1 — Environment (Colab A100 recommended)

Pin torch first if needed (same restart gotcha as the other notebooks).
`tinycudann` is optional — we ship a pure-PyTorch multi-res hash fallback so the
notebook runs without CUDA extensions.


In [ ]:
# 1.0 — Drive + GPU check
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab — skipping Drive mount")

import torch, sys
assert torch.cuda.is_available() or True, "GPU recommended"
print(f"torch {torch.__version__} | cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


### 1.0b — Torch pin for Stage-2 (required if `FLOW_PRIOR_MODE="stage2"`)

`colab_compiled_pkgs.tar.gz` / kaolin are built for **torch 2.5.1+cu121** (same as `mvsam3d_stage2LoRA`).

**Python must be 3.11 or 3.12** — torch 2.5.1 / torchvision 0.20.1 have **no Python 3.13 wheels**.
If you see `py=3.13.x`, go to **Runtime → Change runtime type → Runtime version** and pin
e.g. **2026.07** or **2026.04** (Python 3.12), then reconnect and re-run from §1.

After a successful pin: **Runtime → Restart session**, then continue from §1.1 (do not re-pin).
Wrong torch → `kaolin/_C.so: undefined symbol ...`.

If this cell pins torch, **Runtime → Restart session**, then continue from §1.1 (do not re-pin).


In [ ]:
# 1.0b — Pin torch 2.5.1 for Stage-2 ABI (kaolin / pytorch3d / flash-attn)
import subprocess, sys
import torch

TARGET_TORCH = "2.5.1"
CU_INDEX = "https://download.pytorch.org/whl/cu121"
print(f"PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}  py={sys.version.split()[0]}")

if sys.version_info >= (3, 13):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} cannot install torch==2.5.1 / torchvision==0.20.1 "
        "(no cp313 wheels; Stage-2 compiled stack needs 3.11/3.12).\n"
        "Fix: Runtime → Change runtime type → Runtime version → pin 2026.07 or 2026.04 "
        "(Python 3.12), Connect, then re-run from §1."
    )

def _pip_cu121(*pkgs):
    # --index-url only (not extra): forces cu121 wheels. Separate calls avoid
    # partial resolves where torch downloads but torchvision hits bare PyPI.
    cmd = [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        *pkgs, "--index-url", CU_INDEX,
    ]
    print(" ", " ".join(pkgs))
    subprocess.check_call(cmd)

if not torch.__version__.startswith(TARGET_TORCH):
    print(f"▶ Downgrading torch {torch.__version__} → {TARGET_TORCH}+cu121 …")
    _pip_cu121("torch==2.5.1")
    _pip_cu121("torchvision==0.20.1")
    _pip_cu121("torchaudio==2.5.1")
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "xformers==0.0.28.post3",
            "--index-url", CU_INDEX,
        ])
    except Exception as e:
        print("⚠ xformers pin skipped:", e)
    import importlib
    importlib.invalidate_caches()
    print("✓ Torch pin pip done. ⚠ Runtime → Restart session, then continue from §1.1 (skip re-running this cell if already 2.5.1).")
else:
    print(f"✓ torch {torch.__version__} matches Stage-2 compiled stack")


In [ ]:
# 1.1 — Lightweight deps (hash fallback needs nothing exotic)
!pip install -q numpy==1.26.4 trimesh matplotlib Pillow tqdm scikit-image PyMCubes einops opencv-python-headless

# Optional: tinycudann for Instant-NGP-speed hash (skip if build fails — we fall back)
# !pip install -q git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
print("✓ core imports")


## 2 — Configuration

Edit the `<--` lines. Two data modes:

| Mode | When |
|------|------|
| `DATA_MODE="neus_bundle"` | `cameras_sphere.npz` (BA-NeuS / sam_baneus export) + **RGB/masks from SatelliteFuse** |
| `DATA_MODE="raw_views"` | Load RGB+masks from `views/` or `satellite/` (poses from DA3 npz or identity stub) |


In [ ]:
# 2.0 — Paths and hyper-parameters  (EDIT the <-- marked lines)
from pathlib import Path
from datetime import datetime, timezone
import os

CASE_NAME   = "va_split"                                          # <--
DATA_MODE   = "neus_bundle"                                       # <-- "neus_bundle" | "raw_views"

# ---- va_split on Drive (source RGB + masks; same as sam_baneus / mvsam3d) ----
DATA_ROOT   = Path("/content/drive/MyDrive/SatelliteFuse/datasets/va_split")  # <--
AERIAL_DIR  = DATA_ROOT / "satellite"
IMG_SUBDIR  = "images"
MASK_SUBDIR = "masks"
if not DATA_ROOT.exists():
    _local = Path("/Users/taiganishida/Documents/GSV-MSV/datasets/va_split")
    if _local.exists():
        DATA_ROOT, AERIAL_DIR = _local, _local / "satellite"

VIEW_DIR   = Path("/content/gfhashba_out") / "views"
DA3_NPZ    = Path("/content/gfhashba_out") / "da3_views" / "da3_output.npz"

# BA-NeuS cameras (normalized sphere frame). Images stay on SatelliteFuse.
NEUS_CASE_DIR   = Path("/content/bundle-adjusting-neus/public_data") / CASE_NAME

def _first_existing(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

# Prefer the validated Stage-1 + Sim3 export. Set None to auto-pick latest SamBAneus run.
SAM_BANEU_RUN = "20260811_182029"  # <-- good alignment (edge-refined); None = latest
_SAM_ROOT = Path("/content/drive/MyDrive/SatelliteFuse/SamBAneus")

def _latest_case_dir(root, case):
    if not Path(root).exists():
        return None
    cands = sorted([p for p in Path(root).iterdir() if p.is_dir() and (p / case).exists()], reverse=True)
    return (cands[0] / case) if cands else None

if SAM_BANEU_RUN:
    BANEUS_META_DIR = _SAM_ROOT / SAM_BANEU_RUN / CASE_NAME
    if not BANEUS_META_DIR.exists():
        print(f"⚠ pinned SamBAneus run missing: {BANEUS_META_DIR} — falling back to latest")
        BANEUS_META_DIR = _latest_case_dir(_SAM_ROOT, CASE_NAME)
else:
    BANEUS_META_DIR = _latest_case_dir(_SAM_ROOT, CASE_NAME)

if BANEUS_META_DIR is None:
    BANEUS_META_DIR = _first_existing(
        Path("/content/drive/MyDrive/baneus_ultrra_results") / CASE_NAME,
        Path("/content/drive/MyDrive/SatelliteFuse/baneus_ultrra_results") / CASE_NAME,
    )

BANEUS_LOCAL = _first_existing(
    Path("/content/sam_baneus_ultrra_out"),
    Path("/content/baneus_ultrra_out"),
) or Path("/content/sam_baneus_ultrra_out")

# Drive export first (pinned/good), then local Colab out, then NeuS public_data.
CAMERAS_NPZ = _first_existing(
    *( [BANEUS_META_DIR / "cameras_sphere.npz"] if BANEUS_META_DIR else [] ),
    NEUS_CASE_DIR / "cameras_sphere.npz",
    BANEUS_LOCAL / "cameras_sphere.npz",
)
SCENE_META_NPZ = _first_existing(
    *( [BANEUS_META_DIR / "baneus_scene_meta.npz"] if BANEUS_META_DIR else [] ),
    BANEUS_LOCAL / "baneus_scene_meta.npz",
    Path("/content/gfhashba_out") / "baneus_scene_meta.npz",
)
NEUS_DEPTH_DIR = NEUS_CASE_DIR / "depth" if (NEUS_CASE_DIR / "depth").is_dir() else None

STAGE1_DIR = _first_existing(
    *( [BANEUS_META_DIR / "stage1"] if BANEUS_META_DIR else [] ),
    BANEUS_LOCAL / "stage1",
    Path("/content/gfhashba_out") / "stage1",
)
STAGE1_OCC_NPZ = _first_existing(
    *( [BANEUS_META_DIR / "stage1" / "stage1_occupancy.npz"] if BANEUS_META_DIR else [] ),
    BANEUS_LOCAL / "stage1" / "stage1_occupancy.npz",
    Path("/content/gfhashba_out") / "stage1" / "stage1_occupancy.npz",
)
STAGE1_FRAME_NPZ = _first_existing(
    *( [BANEUS_META_DIR / "stage1" / "stage1_frame.npz"] if BANEUS_META_DIR else [] ),
    BANEUS_LOCAL / "stage1" / "stage1_frame.npz",
    Path("/content/gfhashba_out") / "stage1" / "stage1_frame.npz",
)

# Training resolution — must match cameras_sphere export (auto from baneus_scene_meta.npz)
TRAIN_W = TRAIN_H = None   # None → read meta train_w/train_h, else 768×576

OUTPUT_DIR = Path("/content/gfhashba_out"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TS     = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
DRIVE_OUT  = Path("/content/drive/MyDrive/SatelliteFuse/GFhashBA") / RUN_TS / CASE_NAME

# ---- model ----
HASH_N_LEVELS     = 16
HASH_N_FEATURES   = 2
HASH_LOG2_SIZE    = 19
HASH_BASE_RES     = 16
HASH_MAX_RES      = 2048
SDF_HIDDEN        = 64
SDF_N_LAYERS      = 2
BG_HIDDEN         = 64

# ---- annealing (component 4) ----
ANNEAL_ITERS      = 8000
ANNEAL_START_LVL  = 1.0

# ---- training ----
END_ITER          = 30000
BATCH_RAYS        = 1024
N_SAMPLES         = 64
N_BG_SAMPLES      = 32
LR_SDF            = 1e-2
LR_POSE           = 1e-3
LR_COLOR          = 1e-2
WARMUP_END        = 1000
POSE_START_ITER   = max(200, WARMUP_END // 5)

# ---- loss weights ----
W_PHOTO           = 1.0
W_EIKONAL         = 0.15
W_MASK            = 0.5
W_DEPTH           = 0.1
W_OCC_PRIOR       = 0.5
W_FLOW            = 0.0    # unused; schedule uses W_FLOW_START/END
# Stage-2 vs photo: W_PHOTO=1 every step; flow fires only on adaptive cadence.
# Early: strong flow as basin-finder / smoother (photo landscape is ill-behaved).
# Late: cosine handoff so photo owns fidelity near the minimum.
W_FLOW_START      = 0.55   # per-call weight while held (strong early prior pull)
W_FLOW_END        = 0.06   # late floor on blind spots — residual, not zero
FLOW_HOLD_FRAC    = 0.35   # hold W_START for first 35% of main train (~10.5k/30k)
FLOW_ANNEAL_ITERS = END_ITER  # hold+decay horizon = full main train
# Stage-2 reverse_fn is expensive; cadence is adaptive (see flow_call_period).
# FLOW_EVERY is only a fallback / log label — not a fixed stride.
FLOW_EVERY        = 15        # fallback if adaptive helper missing
PROXY_RES         = 64        # dense distill grid; early calls use 32 for speed
OCC_PRIOR_N       = 4096
VIS_MAX_VIEWS     = 8

# ---- Stage-1 prefit ----
USE_STAGE1_PREFIT = True
SDF_PREFIT_ITERS  = 2000  # was 8000; MSE already ~0 by ~500 (diminishing returns)
SDF_PREFIT_BATCH  = 8192
SDF_PREFIT_LR     = 1e-2

# ---- generative prior ----
# "off" | "occupancy" | "stage2" (real MV-SAM3D) | "stub" | "external"
USE_FLOW_PRIOR    = True          # <-- Stage-2 velocity distill ON (needs §6.0a + MV-SAM3D)
FLOW_PRIOR_MODE   = "stage2"      # <-- real MV-SAM3D Stage-2 SLAT velocity (see §6.0a)
STAGE2_LORA_DIR   = Path("/content/drive/MyDrive/SatelliteFuse/Stage2LoRA/20260813_181113/va_split/lora_adapter")  # <-- or None
STAGE2_TEACHER_STEPS = 25
STAGE2_MICRO_N   = 8
# Prefer SamBAneus-exported Stage-1 coords (same structure as prefit) over re-running SS.
USE_EXPORTED_STAGE1_COORDS = True  # <-- keep True unless debugging fresh SS
# False = Stage-2 only on blind spots (1−V) — default after prefit.
# True  = also distill visible voxels (smoothing A/B).
FLOW_PRIOR_ON_VISIBLE = False  # <-- default gated; True = visible+unseen A/B
FLOW_COND_PROMPT  = "a building"
FLOW_ODE_ETA      = 0.5

# ---- dual-field mask routing ----
HARD_MASK_ROUTE   = True

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CASE={CASE_NAME} | mode={DATA_MODE} | device={DEVICE}")
print(f"DATA_ROOT={DATA_ROOT} | aerial images={AERIAL_DIR / IMG_SUBDIR}")
print(f"SAM_BANEU_RUN={SAM_BANEU_RUN} | BANEUS_META_DIR={BANEUS_META_DIR}")
print(f"CAMERAS_NPZ={CAMERAS_NPZ} | meta={SCENE_META_NPZ}")
print(f"STAGE1_OCC={STAGE1_OCC_NPZ} | STAGE1_DIR={STAGE1_DIR}")
print(f"FLOW_PRIOR_MODE={FLOW_PRIOR_MODE} | USE_FLOW_PRIOR={USE_FLOW_PRIOR}")
print(f"W_FLOW hold {FLOW_HOLD_FRAC:.0%} @ {W_FLOW_START} then cosine → {W_FLOW_END} over {FLOW_ANNEAL_ITERS}")
print(f"FLOW_PRIOR_ON_VISIBLE={FLOW_PRIOR_ON_VISIBLE} | STAGE2_LORA_DIR={STAGE2_LORA_DIR}")
print(f"USE_EXPORTED_STAGE1_COORDS={USE_EXPORTED_STAGE1_COORDS}")
print(f"prefit={USE_STAGE1_PREFIT}x{SDF_PREFIT_ITERS} | W_OCC={W_OCC_PRIOR} | W_EIK={W_EIKONAL} | anneal={ANNEAL_ITERS}")
print(f"OUTPUT_DIR={OUTPUT_DIR} | hash levels={HASH_N_LEVELS} | anneal={ANNEAL_ITERS}")

### 2.1 — Write `gfhashba_align.py` (Colab self-contained)

No local tests required. Dumps Sim3 / densify / mesh-health helpers to disk so
§3.5b / §6.0a / §7 can import them. Re-run after uploading a new notebook.

In [ ]:
# 2.1 — Ensure gfhashba_align.py is on disk (Colab)
from pathlib import Path
import sys, base64, importlib

_ALIGN_B64 = """IiIiR0YtSGFzaEJBIGZyYW1lIG1hdGgg4oCUIHNoYXJlZCBieSBub3RlYm9vayArIGxvY2FsIHVuaXQgdGVzdHMuCgpDYW5vbmljYWwgKFN0YWdlLTEgLyBTTEFUKSDihpIgREEzIHdvcmxkIChTaW0zKSDihpIgTmV1UyB1bml0IHNwaGVyZSBbLTEsMV1eMwoKTmV2ZXIgQUFCQi1zdHJldGNoIFNMQVQgaW5kaWNlcyBpbnRvIHRoZSBOZXVTIGN1YmUuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBNYXBwaW5nLCBPcHRpb25hbAoKaW1wb3J0IG51bXB5IGFzIG5wCgp0cnk6CiAgICBpbXBvcnQgdG9yY2gKICAgIEhBU19UT1JDSCA9IFRydWUKZXhjZXB0IEltcG9ydEVycm9yOiAgIyBwcmFnbWE6IG5vIGNvdmVyCiAgICBIQVNfVE9SQ0ggPSBGYWxzZQogICAgdG9yY2ggPSBOb25lICAjIHR5cGU6IGlnbm9yZQoKCkBkYXRhY2xhc3MKY2xhc3MgQWxpZ25TaW0zOgogICAgTV9jYW5vbmljYWxfdG9fZGEzX3dvcmxkOiBucC5uZGFycmF5CiAgICB3b3JsZF9hbGlnbl9zOiBmbG9hdAogICAgd29ybGRfYWxpZ25fUjogbnAubmRhcnJheQogICAgd29ybGRfYWxpZ25fdDogbnAubmRhcnJheQogICAgY2VudGVyOiBucC5uZGFycmF5CiAgICByYWRpdXM6IGZsb2F0CiAgICB2b3hlbF9ncmlkOiBpbnQgPSA2NAoKICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGlkZW50aXR5KGNscywgdm94ZWxfZ3JpZDogaW50ID0gNjQpIC0+ICJBbGlnblNpbTMiOgogICAgICAgIHJldHVybiBjbHMoCiAgICAgICAgICAgIE1fY2Fub25pY2FsX3RvX2RhM193b3JsZD1ucC5leWUoNCwgZHR5cGU9bnAuZmxvYXQ2NCksCiAgICAgICAgICAgIHdvcmxkX2FsaWduX3M9MS4wLAogICAgICAgICAgICB3b3JsZF9hbGlnbl9SPW5wLmV5ZSgzLCBkdHlwZT1ucC5mbG9hdDY0KSwKICAgICAgICAgICAgd29ybGRfYWxpZ25fdD1ucC56ZXJvcygzLCBkdHlwZT1ucC5mbG9hdDY0KSwKICAgICAgICAgICAgY2VudGVyPW5wLnplcm9zKDMsIGR0eXBlPW5wLmZsb2F0NjQpLAogICAgICAgICAgICByYWRpdXM9MS4wLAogICAgICAgICAgICB2b3hlbF9ncmlkPWludCh2b3hlbF9ncmlkKSwKICAgICAgICApCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgZnJvbV9tYXBwaW5nKGNscywgZDogTWFwcGluZ1tzdHIsIEFueV0pIC0+ICJBbGlnblNpbTMiOgogICAgICAgIGRlZiBfbnAoa2V5LCBkZWZhdWx0KToKICAgICAgICAgICAgaWYga2V5IG5vdCBpbiBkOgogICAgICAgICAgICAgICAgcmV0dXJuIG5wLmFzYXJyYXkoZGVmYXVsdCwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICAgICAgdiA9IGRba2V5XQogICAgICAgICAgICBpZiBIQVNfVE9SQ0ggYW5kIGlzaW5zdGFuY2UodiwgdG9yY2guVGVuc29yKToKICAgICAgICAgICAgICAgIHYgPSB2LmRldGFjaCgpLmNwdSgpLm51bXB5KCkKICAgICAgICAgICAgcmV0dXJuIG5wLmFzYXJyYXkodiwgZHR5cGU9bnAuZmxvYXQ2NCkKCiAgICAgICAgcmV0dXJuIGNscygKICAgICAgICAgICAgTV9jYW5vbmljYWxfdG9fZGEzX3dvcmxkPV9ucCgiTV9jYW5vbmljYWxfdG9fZGEzX3dvcmxkIiwgbnAuZXllKDQpKSwKICAgICAgICAgICAgd29ybGRfYWxpZ25fcz1mbG9hdChkLmdldCgid29ybGRfYWxpZ25fcyIsIDEuMCkpLAogICAgICAgICAgICB3b3JsZF9hbGlnbl9SPV9ucCgid29ybGRfYWxpZ25fUiIsIG5wLmV5ZSgzKSksCiAgICAgICAgICAgIHdvcmxkX2FsaWduX3Q9X25wKCJ3b3JsZF9hbGlnbl90IiwgbnAuemVyb3MoMykpLAogICAgICAgICAgICBjZW50ZXI9X25wKCJjZW50ZXIiLCBucC56ZXJvcygzKSksCiAgICAgICAgICAgIHJhZGl1cz1mbG9hdChkLmdldCgicmFkaXVzIiwgMS4wKSksCiAgICAgICAgICAgIHZveGVsX2dyaWQ9aW50KGQuZ2V0KCJ2b3hlbF9ncmlkIiwgNjQpKSwKICAgICAgICApCgogICAgZGVmIGFzX3RvcmNoX2RpY3Qoc2VsZik6CiAgICAgICAgaWYgbm90IEhBU19UT1JDSDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJ0b3JjaCByZXF1aXJlZCIpCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgIk1fY2Fub25pY2FsX3RvX2RhM193b3JsZCI6IHRvcmNoLnRlbnNvcihzZWxmLk1fY2Fub25pY2FsX3RvX2RhM193b3JsZCwgZHR5cGU9dG9yY2guZmxvYXQzMiksCiAgICAgICAgICAgICJ3b3JsZF9hbGlnbl9zIjogZmxvYXQoc2VsZi53b3JsZF9hbGlnbl9zKSwKICAgICAgICAgICAgIndvcmxkX2FsaWduX1IiOiB0b3JjaC50ZW5zb3Ioc2VsZi53b3JsZF9hbGlnbl9SLCBkdHlwZT10b3JjaC5mbG9hdDMyKSwKICAgICAgICAgICAgIndvcmxkX2FsaWduX3QiOiB0b3JjaC50ZW5zb3Ioc2VsZi53b3JsZF9hbGlnbl90LCBkdHlwZT10b3JjaC5mbG9hdDMyKSwKICAgICAgICAgICAgImNlbnRlciI6IHRvcmNoLnRlbnNvcihzZWxmLmNlbnRlciwgZHR5cGU9dG9yY2guZmxvYXQzMiksCiAgICAgICAgICAgICJyYWRpdXMiOiBmbG9hdChzZWxmLnJhZGl1cyksCiAgICAgICAgICAgICJ2b3hlbF9ncmlkIjogaW50KHNlbGYudm94ZWxfZ3JpZCksCiAgICAgICAgfQoKCmRlZiB2b3hlbF9pZHhfdG9fY2Fub25pY2FsKGNvb3Jkc194eXo6IG5wLm5kYXJyYXksIHZveGVsX2dyaWQ6IGludCA9IDY0KSAtPiBucC5uZGFycmF5OgogICAgYyA9IG5wLmFzYXJyYXkoY29vcmRzX3h5eiwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIGMubmRpbSAhPSAyIG9yIGMuc2hhcGVbMV0gbm90IGluICgzLCA0KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiY29vcmRzIGV4cGVjdGVkIChOLDN8NCksIGdvdCB7Yy5zaGFwZX0iKQogICAgaWYgYy5zaGFwZVsxXSA9PSA0OgogICAgICAgIGMgPSBjWzosIDE6NF0KICAgIHJldHVybiBjIC8gZmxvYXQodm94ZWxfZ3JpZCkgLSAwLjUKCgpkZWYgY2Fub25pY2FsX3RvX25ldXNfbm9ybSh4X2NhbjogbnAubmRhcnJheSwgYWxpZ246IEFsaWduU2ltMykgLT4gbnAubmRhcnJheToKICAgIHggPSBucC5hc2FycmF5KHhfY2FuLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgb25lcyA9IG5wLm9uZXMoKHguc2hhcGVbMF0sIDEpLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgeF91bmFsID0gKG5wLmNvbmNhdGVuYXRlKFt4LCBvbmVzXSwgYXhpcz0tMSkgQCBhbGlnbi5NX2Nhbm9uaWNhbF90b19kYTNfd29ybGQpWzosIDozXQogICAgeF93b3JsZCA9IGZsb2F0KGFsaWduLndvcmxkX2FsaWduX3MpICogKHhfdW5hbCBAIGFsaWduLndvcmxkX2FsaWduX1IpICsgYWxpZ24ud29ybGRfYWxpZ25fdAogICAgcmV0dXJuICh4X3dvcmxkIC0gYWxpZ24uY2VudGVyKSAvIGZsb2F0KGFsaWduLnJhZGl1cykKCgpkZWYgbmV1c19ub3JtX3RvX2Nhbm9uaWNhbCh4X25vcm06IG5wLm5kYXJyYXksIGFsaWduOiBBbGlnblNpbTMpIC0+IG5wLm5kYXJyYXk6CiAgICB4X25vcm0gPSBucC5hc2FycmF5KHhfbm9ybSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIHhfd29ybGQgPSB4X25vcm0gKiBmbG9hdChhbGlnbi5yYWRpdXMpICsgYWxpZ24uY2VudGVyCiAgICB4X3VuYWwgPSAoeF93b3JsZCAtIGFsaWduLndvcmxkX2FsaWduX3QpIEAgYWxpZ24ud29ybGRfYWxpZ25fUi5UIC8gZmxvYXQoYWxpZ24ud29ybGRfYWxpZ25fcykKICAgIG9uZXMgPSBucC5vbmVzKCh4X3VuYWwuc2hhcGVbMF0sIDEpLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgTV9pbnYgPSBucC5saW5hbGcuaW52KGFsaWduLk1fY2Fub25pY2FsX3RvX2RhM193b3JsZCkKICAgIHJldHVybiAobnAuY29uY2F0ZW5hdGUoW3hfdW5hbCwgb25lc10sIGF4aXM9LTEpIEAgTV9pbnYpWzosIDozXQoKCmRlZiB2b3hlbF9pZHhfdG9fbmV1cyhjb29yZHNfeHl6OiBucC5uZGFycmF5LCBhbGlnbjogQWxpZ25TaW0zKSAtPiBucC5uZGFycmF5OgogICAgcmV0dXJuIGNhbm9uaWNhbF90b19uZXVzX25vcm0odm94ZWxfaWR4X3RvX2Nhbm9uaWNhbChjb29yZHNfeHl6LCBhbGlnbi52b3hlbF9ncmlkKSwgYWxpZ24pCgoKZGVmIGFhYmJfc3RyZXRjaF90b19ncmlkKGNvb3Jkc194eXo6IG5wLm5kYXJyYXksIHJlczogaW50KSAtPiBucC5uZGFycmF5OgogICAgIiIiQlVHR1kgb2xkIGRlbnNpZnkuIFJldHVybnMgKE4sMykgaW50ZWdlciB4LHksei4iIiIKICAgIGMgPSBucC5hc2FycmF5KGNvb3Jkc194eXosIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBpZiBjLnNoYXBlWzFdID09IDQ6CiAgICAgICAgYyA9IGNbOiwgMTo0XQogICAgY21pbiwgY21heCA9IGMubWluKDApLCBjLm1heCgwKQogICAgc3BhbiA9IG5wLm1heGltdW0oY21heCAtIGNtaW4sIDEuMCkKICAgIGMwMSA9IChjIC0gY21pbikgLyBzcGFuCiAgICByZXR1cm4gbnAucm91bmQoYzAxICogKHJlcyAtIDEpKS5hc3R5cGUobnAuaW50NjQpLmNsaXAoMCwgcmVzIC0gMSkKCgpkZWYgc2ltM19zb2Z0X3NwbGF0X3RvX2dyaWQoCiAgICBjb29yZHNfeHl6OiBucC5uZGFycmF5LAogICAgd2VpZ2h0czogT3B0aW9uYWxbbnAubmRhcnJheV0sCiAgICBhbGlnbjogQWxpZ25TaW0zLAogICAgcmVzOiBpbnQgPSA2NCwKKSAtPiBucC5uZGFycmF5OgogICAgeF9ub3JtID0gdm94ZWxfaWR4X3RvX25ldXMoY29vcmRzX3h5eiwgYWxpZ24pCiAgICBnID0gKHhfbm9ybSArIDEuMCkgKiAwLjUgKiAocmVzIC0gMSkKICAgIGluc2lkZSA9IG5wLmFsbChucC5hYnMoeF9ub3JtKSA8PSAxLjA1LCBheGlzPTEpCiAgICBnID0gZ1tpbnNpZGVdCiAgICBpZiB3ZWlnaHRzIGlzIE5vbmU6CiAgICAgICAgdyA9IG5wLm9uZXMoZy5zaGFwZVswXSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGVsc2U6CiAgICAgICAgdyA9IG5wLmFzYXJyYXkod2VpZ2h0cywgZHR5cGU9bnAuZmxvYXQ2NClbaW5zaWRlXQogICAgICAgIHcgPSB3IC8gbWF4KGZsb2F0KHcubWVhbigpKSwgMWUtOCkKICAgIHZvbCA9IG5wLnplcm9zKChyZXMsIHJlcywgcmVzKSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIGcuc2l6ZSA9PSAwOgogICAgICAgIHJldHVybiB2b2wKICAgIHgsIHksIHogPSBnWzosIDBdLCBnWzosIDFdLCBnWzosIDJdCiAgICB4MCA9IG5wLmZsb29yKHgpLmFzdHlwZShucC5pbnQ2NCkKICAgIHkwID0gbnAuZmxvb3IoeSkuYXN0eXBlKG5wLmludDY0KQogICAgejAgPSBucC5mbG9vcih6KS5hc3R5cGUobnAuaW50NjQpCiAgICBmb3IgZHosIHd6IGluICgoMCwgMS4wIC0gKHogLSB6MCkpLCAoMSwgKHogLSB6MCkpKToKICAgICAgICBmb3IgZHksIHd5IGluICgoMCwgMS4wIC0gKHkgLSB5MCkpLCAoMSwgKHkgLSB5MCkpKToKICAgICAgICAgICAgZm9yIGR4LCB3eCBpbiAoKDAsIDEuMCAtICh4IC0geDApKSwgKDEsICh4IC0geDApKSk6CiAgICAgICAgICAgICAgICBpeCA9IG5wLmNsaXAoeDAgKyBkeCwgMCwgcmVzIC0gMSkKICAgICAgICAgICAgICAgIGl5ID0gbnAuY2xpcCh5MCArIGR5LCAwLCByZXMgLSAxKQogICAgICAgICAgICAgICAgaXogPSBucC5jbGlwKHowICsgZHosIDAsIHJlcyAtIDEpCiAgICAgICAgICAgICAgICBucC5hZGQuYXQodm9sLCAoaXosIGl5LCBpeCksIG5wLmFzYXJyYXkod3ogKiB3eSAqIHd4ICogdywgZHR5cGU9bnAuZmxvYXQ2NCkpCiAgICBtID0gZmxvYXQodm9sLm1heCgpKQogICAgaWYgbSA+IDA6CiAgICAgICAgdm9sIC89IG0KICAgIHJldHVybiB2b2wKCgpkZWYgZ3JpZF9pb3UoYTogbnAubmRhcnJheSwgYjogbnAubmRhcnJheSwgdGhyX2E6IGZsb2F0ID0gMC4xLCB0aHJfYjogZmxvYXQgPSAwLjUpIC0+IGZsb2F0OgogICAgYWEsIGJiID0gYSA+IHRocl9hLCBiID4gdGhyX2IKICAgIGludGVyID0gbnAubG9naWNhbF9hbmQoYWEsIGJiKS5zdW0oKQogICAgdW5pb24gPSBucC5sb2dpY2FsX29yKGFhLCBiYikuc3VtKCkKICAgIHJldHVybiBmbG9hdChpbnRlciAvIG1heCh1bmlvbiwgMSkpCgoKZGVmIGdyaWRfY29ycihhOiBucC5uZGFycmF5LCBiOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgIGFhID0gYS5yYXZlbCgpLmFzdHlwZShucC5mbG9hdDY0KQogICAgYmIgPSBiLnJhdmVsKCkuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICBhYSA9IChhYSAtIGFhLm1lYW4oKSkgLyAoYWEuc3RkKCkgKyAxZS04KQogICAgYmIgPSAoYmIgLSBiYi5tZWFuKCkpIC8gKGJiLnN0ZCgpICsgMWUtOCkKICAgIHJldHVybiBmbG9hdCgoYWEgKiBiYikubWVhbigpKQoKCmRlZiBzZGZfdm9sdW1lX2hlYWx0aCh2b2w6IG5wLm5kYXJyYXksIGxldmVsOiBmbG9hdCA9IDAuMCkgLT4gZGljdDoKICAgIHZvbCA9IG5wLmFzYXJyYXkodm9sLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgaW5zaWRlID0gdm9sIDwgbGV2ZWwKICAgIGZyYWNfaW5zaWRlID0gZmxvYXQoaW5zaWRlLm1lYW4oKSkKICAgIHZtaW4sIHZtYXggPSBmbG9hdCh2b2wubWluKCkpLCBmbG9hdCh2b2wubWF4KCkpCiAgICBjcm9zc2VzID0gYm9vbCh2bWluIDwgbGV2ZWwgPCB2bWF4KQogICAgcmV0dXJuIHsKICAgICAgICAidm1pbiI6IHZtaW4sCiAgICAgICAgInZtYXgiOiB2bWF4LAogICAgICAgICJsZXZlbCI6IGZsb2F0KGxldmVsKSwKICAgICAgICAiY3Jvc3Nlc196ZXJvIjogY3Jvc3NlcywKICAgICAgICAiZnJhY19pbnNpZGUiOiBmcmFjX2luc2lkZSwKICAgICAgICAiZnJhY19vdXRzaWRlIjogMS4wIC0gZnJhY19pbnNpZGUsCiAgICAgICAgImR5bmFtaWNfcmFuZ2UiOiB2bWF4IC0gdm1pbiwKICAgICAgICAib2tfc2lnbl9iYWxhbmNlIjogY3Jvc3NlcyBhbmQgKDAuMDEgPD0gZnJhY19pbnNpZGUgPD0gMC44NSksCiAgICAgICAgIm9rX3JhbmdlIjogKHZtYXggLSB2bWluKSA+IDFlLTMsCiAgICB9CgoKZGVmIG1lc2hfaGVhbHRoKHZlcnRpY2VzOiBucC5uZGFycmF5LCBmYWNlczogbnAubmRhcnJheSwgb2NjX2Jib3g6IE9wdGlvbmFsW25wLm5kYXJyYXldID0gTm9uZSkgLT4gZGljdDoKICAgIHYgPSBucC5hc2FycmF5KHZlcnRpY2VzLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgZiA9IG5wLmFzYXJyYXkoZmFjZXMsIGR0eXBlPW5wLmludDY0KQogICAgbl92LCBuX2YgPSBsZW4odiksIGxlbihmKQogICAgaWYgbl92ID09IDAgb3Igbl9mID09IDA6CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgIm5fdmVydHMiOiBuX3YsICJuX2ZhY2VzIjogbl9mLCAibl9jb21wb25lbnRzIjogMCwgImxhcmdlc3RfZnJhYyI6IDAuMCwKICAgICAgICAgICAgImJib3hfbWluIjogTm9uZSwgImJib3hfbWF4IjogTm9uZSwgIm9rIjogRmFsc2UsICJyZWFzb24iOiAiZW1wdHkgbWVzaCIsCiAgICAgICAgfQogICAgcGFyZW50ID0gbnAuYXJhbmdlKG5fdikKCiAgICBkZWYgZmluZChpOiBpbnQpIC0+IGludDoKICAgICAgICB3aGlsZSBwYXJlbnRbaV0gIT0gaToKICAgICAgICAgICAgcGFyZW50W2ldID0gcGFyZW50W3BhcmVudFtpXV0KICAgICAgICAgICAgaSA9IGludChwYXJlbnRbaV0pCiAgICAgICAgcmV0dXJuIGkKCiAgICBkZWYgdW5pb24oYTogaW50LCBiOiBpbnQpIC0+IE5vbmU6CiAgICAgICAgcmEsIHJiID0gZmluZChhKSwgZmluZChiKQogICAgICAgIGlmIHJhICE9IHJiOgogICAgICAgICAgICBwYXJlbnRbcmJdID0gcmEKCiAgICBmb3IgdHJpIGluIGY6CiAgICAgICAgdW5pb24oaW50KHRyaVswXSksIGludCh0cmlbMV0pKQogICAgICAgIHVuaW9uKGludCh0cmlbMV0pLCBpbnQodHJpWzJdKSkKCiAgICB1c2VkID0gbnAudW5pcXVlKGYucmF2ZWwoKSkKICAgIHJvb3RzID0gW2ZpbmQoaW50KGkpKSBmb3IgaSBpbiB1c2VkXQogICAgY250ID0gQ291bnRlcihyb290cykKICAgIG5fY29tcCA9IGxlbihjbnQpCiAgICBsYXJnZXN0ID0gbWF4KGNudC52YWx1ZXMoKSkgaWYgY250IGVsc2UgMAogICAgbGFyZ2VzdF9mcmFjID0gZmxvYXQobGFyZ2VzdCAvIG1heChsZW4odXNlZCksIDEpKQogICAgYmJfbWluLCBiYl9tYXggPSB2Lm1pbigwKSwgdi5tYXgoMCkKICAgIGJiX2V4dCA9IGJiX21heCAtIGJiX21pbgogICAgb2sgPSBuX2NvbXAgPD0gMjAgYW5kIGxhcmdlc3RfZnJhYyA+PSAwLjUgYW5kIG5fZiA+PSA1MDAKICAgIHJlYXNvbiA9ICJvayIKICAgIGlmIG5fY29tcCA+IDIwOgogICAgICAgIHJlYXNvbiA9IGYidG9vIG1hbnkgY29tcG9uZW50cyAoe25fY29tcH0pIOKAlCBmbG9hdGluZyBkZWJyaXMiCiAgICBlbGlmIGxhcmdlc3RfZnJhYyA8IDAuNToKICAgICAgICByZWFzb24gPSBmImxhcmdlc3QgY29tcG9uZW50IG9ubHkge2xhcmdlc3RfZnJhYzouMiV9IG9mIHZlcnRzIgogICAgZWxpZiBuX2YgPCA1MDA6CiAgICAgICAgcmVhc29uID0gInRvbyBmZXcgZmFjZXMiCiAgICBvdXQgPSB7CiAgICAgICAgIm5fdmVydHMiOiBuX3YsICJuX2ZhY2VzIjogbl9mLCAibl9jb21wb25lbnRzIjogbl9jb21wLCAibGFyZ2VzdF9mcmFjIjogbGFyZ2VzdF9mcmFjLAogICAgICAgICJiYm94X21pbiI6IGJiX21pbiwgImJib3hfbWF4IjogYmJfbWF4LCAiYmJveF9leHRlbnQiOiBiYl9leHQsICJvayI6IG9rLCAicmVhc29uIjogcmVhc29uLAogICAgfQogICAgaWYgb2NjX2Jib3ggaXMgbm90IE5vbmU6CiAgICAgICAgb2NjX21pbiwgb2NjX21heCA9IG9jY19iYm94WzBdLCBvY2NfYmJveFsxXQogICAgICAgIG1uID0gbnAubWF4aW11bShiYl9taW4sIG9jY19taW4pCiAgICAgICAgbXggPSBucC5taW5pbXVtKGJiX21heCwgb2NjX21heCkKICAgICAgICBpbnRlciA9IGZsb2F0KG5wLm1heGltdW0oMC4wLCBteCAtIG1uKS5wcm9kKCkpCiAgICAgICAgdW5pb25fdiA9IGZsb2F0KGJiX2V4dC5wcm9kKCkgKyBucC5tYXhpbXVtKDAuMCwgb2NjX21heCAtIG9jY19taW4pLnByb2QoKSAtIGludGVyKQogICAgICAgIG91dFsiYmJveF9pb3VfdnNfb2NjIl0gPSBpbnRlciAvIG1heCh1bmlvbl92LCAxZS0xMikKICAgICAgICBvdXRbIm9rX2Jib3hfb3ZlcmxhcCJdID0gb3V0WyJiYm94X2lvdV92c19vY2MiXSA+PSAwLjE1CiAgICAgICAgaWYgb3V0WyJvayJdIGFuZCBub3Qgb3V0WyJva19iYm94X292ZXJsYXAiXToKICAgICAgICAgICAgb3V0WyJvayJdID0gRmFsc2UKICAgICAgICAgICAgb3V0WyJyZWFzb24iXSA9IGYibWVzaCBiYm94IElvVSB2cyBPQ0M9e291dFsnYmJveF9pb3VfdnNfb2NjJ106LjNmfSB0b28gbG93IgogICAgcmV0dXJuIG91dAoKCmRlZiBvY2NfYmJveF9mcm9tX2dyaWQob2NjOiBucC5uZGFycmF5LCB0aHI6IGZsb2F0ID0gMC41KSAtPiBPcHRpb25hbFtucC5uZGFycmF5XToKICAgIG9jYyA9IG5wLmFzYXJyYXkob2NjKQogICAgaWR4ID0gbnAuYXJnd2hlcmUob2NjID4gdGhyKQogICAgaWYgbGVuKGlkeCkgPT0gMDoKICAgICAgICByZXR1cm4gTm9uZQogICAgcmVzID0gb2NjLnNoYXBlWzBdCiAgICB6eXggPSBpZHguYXN0eXBlKG5wLmZsb2F0NjQpCiAgICB4eXogPSBucC5zdGFjayhbCiAgICAgICAgenl4WzosIDJdIC8gKHJlcyAtIDEpICogMiAtIDEsCiAgICAgICAgenl4WzosIDFdIC8gKHJlcyAtIDEpICogMiAtIDEsCiAgICAgICAgenl4WzosIDBdIC8gKHJlcyAtIDEpICogMiAtIDEsCiAgICBdLCBheGlzPS0xKQogICAgcmV0dXJuIG5wLnN0YWNrKFt4eXoubWluKDApLCB4eXoubWF4KDApXSwgYXhpcz0wKQoKCmRlZiB0b3JjaF9jYW5vbmljYWxfdG9fbmV1cyh4X2NhbiwgYWxpZ25fZGljdCk6CiAgICBpZiBub3QgSEFTX1RPUkNIOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigidG9yY2ggcmVxdWlyZWQiKQogICAgYSA9IEFsaWduU2ltMy5mcm9tX21hcHBpbmcoYWxpZ25fZGljdCkKICAgIHRkID0gYS5hc190b3JjaF9kaWN0KCkKICAgIGRldiwgZHR5cGUgPSB4X2Nhbi5kZXZpY2UsIHhfY2FuLmR0eXBlCiAgICBvbmVzID0gdG9yY2gub25lcyh4X2Nhbi5zaGFwZVswXSwgMSwgZGV2aWNlPWRldiwgZHR5cGU9ZHR5cGUpCiAgICB4X3VuYWwgPSAodG9yY2guY2F0KFt4X2Nhbiwgb25lc10sIGRpbT0tMSkgQCB0ZFsiTV9jYW5vbmljYWxfdG9fZGEzX3dvcmxkIl0udG8oZGV2LCBkdHlwZSkpWzosIDozXQogICAgeF93b3JsZCA9IGZsb2F0KHRkWyJ3b3JsZF9hbGlnbl9zIl0pICogKHhfdW5hbCBAIHRkWyJ3b3JsZF9hbGlnbl9SIl0udG8oZGV2LCBkdHlwZSkpICsgdGRbIndvcmxkX2FsaWduX3QiXS50byhkZXYsIGR0eXBlKQogICAgcmV0dXJuICh4X3dvcmxkIC0gdGRbImNlbnRlciJdLnRvKGRldiwgZHR5cGUpKSAvIGZsb2F0KHRkWyJyYWRpdXMiXSkKCgpkZWYgdG9yY2hfbmV1c190b19jYW5vbmljYWwoeF9ub3JtLCBhbGlnbl9kaWN0KToKICAgIGlmIG5vdCBIQVNfVE9SQ0g6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJ0b3JjaCByZXF1aXJlZCIpCiAgICBhID0gQWxpZ25TaW0zLmZyb21fbWFwcGluZyhhbGlnbl9kaWN0KQogICAgdGQgPSBhLmFzX3RvcmNoX2RpY3QoKQogICAgTV9pbnYgPSB0b3JjaC5saW5hbGcuaW52KHRkWyJNX2Nhbm9uaWNhbF90b19kYTNfd29ybGQiXSkKICAgIGRldiwgZHR5cGUgPSB4X25vcm0uZGV2aWNlLCB4X25vcm0uZHR5cGUKICAgIHhfd29ybGQgPSB4X25vcm0gKiBmbG9hdCh0ZFsicmFkaXVzIl0pICsgdGRbImNlbnRlciJdLnRvKGRldiwgZHR5cGUpCiAgICB4X3VuYWwgPSAoeF93b3JsZCAtIHRkWyJ3b3JsZF9hbGlnbl90Il0udG8oZGV2LCBkdHlwZSkpIEAgdGRbIndvcmxkX2FsaWduX1IiXS50byhkZXYsIGR0eXBlKS5UIC8gZmxvYXQodGRbIndvcmxkX2FsaWduX3MiXSkKICAgIG9uZXMgPSB0b3JjaC5vbmVzKHhfdW5hbC5zaGFwZVswXSwgMSwgZGV2aWNlPWRldiwgZHR5cGU9ZHR5cGUpCiAgICByZXR1cm4gKHRvcmNoLmNhdChbeF91bmFsLCBvbmVzXSwgZGltPS0xKSBAIE1faW52LnRvKGRldiwgZHR5cGUpKVs6LCA6M10K"""
_ALIGN_BYTES = base64.b64decode(_ALIGN_B64)
_ALIGN_DESTS = [Path("/content/gfhashba_align.py"), Path.cwd() / "gfhashba_align.py"]

_written = None
for _d in _ALIGN_DESTS:
    try:
        _d.parent.mkdir(parents=True, exist_ok=True)
        if (not _d.exists()) or (_d.stat().st_size != len(_ALIGN_BYTES)):
            _d.write_bytes(_ALIGN_BYTES)
        _written = _d
        break
    except Exception as _e:
        print("skip", _d, ":", _e)

if _written is None:
    raise RuntimeError("Could not write gfhashba_align.py")

_parent = str(_written.parent.resolve())
if _parent not in sys.path:
    sys.path.insert(0, _parent)

import gfhashba_align
importlib.reload(gfhashba_align)
from gfhashba_align import AlignSim3, sdf_volume_health, mesh_health  # noqa: F401
print("✓ gfhashba_align ready at", _written, f"({len(_ALIGN_BYTES)} bytes)")

## 3 — Load posed RGB + masks (+ optional depth)

Loads a BA-NeuS `cameras_sphere.npz` bundle when available so GF-HashBA starts
in the **same normalized unit-sphere frame** as `sam_baneus_ultrra.ipynb`.


In [ ]:
# 3.0 — Dataset helpers (NeuS cameras_sphere.npz or raw stub)
import json
import numpy as np
import torch
from pathlib import Path
from PIL import Image

def load_K_Rt_from_P(P):
    """IDR / BA-NeuS convention: P = K @ [R|t], 3x4."""
    import cv2
    P = np.asarray(P, np.float64)
    if P.shape == (4, 4):
        P = P[:3]
    P = P[:3, :4]
    K, R, t, _, _, _, _ = cv2.decomposeProjectionMatrix(P)
    K = K / K[2, 2]
    pose = np.eye(4, dtype=np.float64)
    pose[:3, :3] = R.T
    pose[:3, 3] = (t[:3] / t[3])[:, 0]   # homogeneous t from OpenCV
    return K[:3, :3].astype(np.float32), pose.astype(np.float32)

def _resolve_train_size(meta_npz=None, train_w=None, train_h=None):
    if train_w and train_h:
        return int(train_w), int(train_h)
    if meta_npz is not None and Path(meta_npz).exists():
        meta = np.load(meta_npz, allow_pickle=True)
        if "train_w" in meta.files and "train_h" in meta.files:
            return int(meta["train_w"]), int(meta["train_h"])
    return 768, 576   # sam_baneus va_split default

def _resize_to_train(img, mask, depth, K, train_w, train_h, scale_k=True):
    """Resize RGB/mask/depth to export resolution; optionally scale intrinsics."""
    import cv2
    h0, w0 = img.shape[:2]
    if (w0, h0) == (train_w, train_h):
        return img, mask, depth, K.astype(np.float32)
    img_u8 = (np.clip(img, 0.0, 1.0) * 255.0).astype(np.uint8)
    img_r = np.asarray(
        Image.fromarray(img_u8).resize((train_w, train_h), Image.LANCZOS), np.float32
    ) / 255.0
    mask_r = cv2.resize(mask.astype(np.float32), (train_w, train_h), interpolation=cv2.INTER_NEAREST)
    depth_r = None
    if depth is not None:
        depth_r = cv2.resize(depth.astype(np.float32), (train_w, train_h), interpolation=cv2.INTER_NEAREST)
    K_r = np.asarray(K, np.float32).copy()
    if scale_k:
        sx, sy = train_w / w0, train_h / h0
        K_r[0, 0] *= sx; K_r[1, 1] *= sy; K_r[0, 2] *= sx; K_r[1, 2] *= sy
    return img_r, mask_r, depth_r, K_r

def _glob_stem(directory, stem):
    directory = Path(directory)
    for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG"):
        p = directory / f"{stem}{ext}"
        if p.exists():
            return p
    matches = sorted(directory.glob(f"{stem}.*"))
    return matches[0] if matches else None

def _read_view_names(cameras_npz, meta_npz=None, images_dir=None, da3_npz=None, n_views=None):
    if meta_npz is not None and Path(meta_npz).exists():
        meta = np.load(meta_npz, allow_pickle=True)
        if "view_names" in meta.files:
            return [str(x) for x in meta["view_names"]]
    txt = Path(cameras_npz).parent / "view_names.txt"
    if txt.exists():
        return [ln.strip() for ln in txt.read_text().splitlines() if ln.strip()]
    if da3_npz is not None and Path(da3_npz).exists() and n_views is not None:
        da3 = np.load(da3_npz, allow_pickle=True)
        stems = [Path(str(p)).stem for p in da3["image_files"]]
        if len(stems) == n_views:
            return stems
    if images_dir is not None and Path(images_dir).exists():
        files = sorted(list(Path(images_dir).glob("*.png")) + list(Path(images_dir).glob("*.jpg")))
        if n_views is None or len(files) == n_views:
            return [f.stem for f in files]
    return None

def load_neus_bundle(cameras_npz, images_dir, masks_dir=None, depth_dir=None,
                     meta_npz=None, view_names=None, da3_npz=None,
                     train_w=None, train_h=None):
    """Cameras from BA-NeuS npz; RGB/masks from SatelliteFuse (or views/)."""
    cameras_npz = Path(cameras_npz)
    images_dir = Path(images_dir)
    masks_dir = Path(masks_dir) if masks_dir else images_dir.parent / "masks"
    depth_dir = Path(depth_dir) if depth_dir else None
    train_w, train_h = _resolve_train_size(meta_npz, train_w, train_h)

    cam = np.load(cameras_npz)
    n = len([k for k in cam.files if k.startswith("world_mat_")])
    if view_names is None:
        view_names = _read_view_names(cameras_npz, meta_npz, images_dir, da3_npz, n)
    if view_names is None or len(view_names) != n:
        raise FileNotFoundError(
            f"Need {n} view_names matching cameras_sphere.npz; got "
            f"{len(view_names) if view_names else 0}. "
            f"Restore baneus_scene_meta.npz from sam_baneus or run §12 export."
        )

    images, masks, depths, Ks, c2ws = [], [], [], [], []
    for i, name in enumerate(view_names):
        P = cam[f"world_mat_{i}"] @ cam[f"scale_mat_{i}"]
        K, c2w = load_K_Rt_from_P(P)
        Ks.append(K); c2ws.append(c2w)

        ip = _glob_stem(images_dir, name)
        if ip is None:
            raise FileNotFoundError(
                f"Missing source image '{name}' under {images_dir} "
                f"(SatelliteFuse satellite/images, not baneus_ultrra_results)"
            )
        img = np.asarray(Image.open(ip).convert("RGB"), np.float32) / 255.0

        mp = _glob_stem(masks_dir, name)
        if mp is not None:
            mimg = Image.open(mp)
            if mimg.mode == "RGBA":
                m = (np.asarray(mimg)[..., 3] > 127).astype(np.float32)
            else:
                m = (np.asarray(mimg.convert("L")) > 127).astype(np.float32)
        else:
            m = np.ones(img.shape[:2], np.float32)
        if m.shape != img.shape[:2]:
            import cv2
            m = cv2.resize(m.astype(np.float32), (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

        dep = None
        dp = depth_dir / f"{i:03d}.npy" if depth_dir is not None else None
        if dp is not None and dp.exists():
            dep = np.load(dp).astype(np.float32)

        img, m, dep, K = _resize_to_train(img, m, dep, K, train_w, train_h, scale_k=False)
        images.append(img); masks.append(m); depths.append(dep); Ks[-1] = K

    if da3_npz is not None and Path(da3_npz).exists():
        da3 = np.load(da3_npz, allow_pickle=True)
        d_stems = [Path(str(p)).stem for p in da3["image_files"]]
        for i, s in enumerate(view_names):
            if depths[i] is not None or s not in d_stems:
                continue
            j = d_stems.index(s)
            dep = da3["depth"][j].astype(np.float32)
            _, _, dep, _ = _resize_to_train(images[i], masks[i], dep, Ks[i], train_w, train_h, scale_k=False)
            depths[i] = dep
        n_dep = sum(d is not None for d in depths)
        if n_dep:
            print(f"Depth: {n_dep}/{n} views (bundle + DA3 fallback)")

    print(f"Train resolution: {train_w}x{train_h} | views: {n}")
    return {
        "images": images, "masks": masks, "depths": depths,
        "Ks": np.stack(Ks), "c2ws": np.stack(c2ws),
        "H": images[0].shape[0], "W": images[0].shape[1], "n_views": n,
        "view_names": list(view_names),
        "scale_mat": cam["scale_mat_0"] if "scale_mat_0" in cam.files else np.eye(4),
    }

def load_raw_views(images_dir, masks_dir, da3_npz=None, max_views=None,
                   train_w=None, train_h=None):
    """Fallback: RGB+masks; poses from DA3 if present else identity stubs."""
    import cv2
    images_dir, masks_dir = Path(images_dir), Path(masks_dir)
    train_w, train_h = _resolve_train_size(None, train_w, train_h)
    files = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")))
    if max_views:
        files = files[:max_views]
    da3 = np.load(da3_npz, allow_pickle=True) if da3_npz is not None and Path(da3_npz).exists() else None
    d_stems = [Path(str(p)).stem for p in da3["image_files"]] if da3 is not None else []

    images, masks, depths, Ks, c2ws, stems = [], [], [], [], [], []
    for f in files:
        img = np.asarray(Image.open(f).convert("RGB"), np.float32) / 255.0
        H, W = img.shape[:2]
        mp = masks_dir / f"{f.stem}.png"
        if mp.exists():
            mimg = Image.open(mp)
            if mimg.mode == "RGBA":
                m = (np.asarray(mimg)[..., 3] > 127).astype(np.float32)
            else:
                m = (np.asarray(mimg.convert("L")) > 127).astype(np.float32)
        else:
            m = np.ones((H, W), np.float32)
        if m.shape != (H, W):
            m = cv2.resize(m.astype(np.float32), (W, H), interpolation=cv2.INTER_NEAREST)
        K = np.array([[W, 0, W / 2], [0, W, H / 2], [0, 0, 1]], np.float32)
        c2w = np.eye(4, dtype=np.float32)
        dep = None
        if da3 is not None and f.stem in d_stems:
            j = d_stems.index(f.stem)
            E = da3["extrinsics"][j]
            w2c = np.eye(4, dtype=np.float64); w2c[:3] = E[:3]
            c2w = np.linalg.inv(w2c).astype(np.float32)
            K = da3["intrinsics"][j].astype(np.float32)
            dep = da3["depth"][j].astype(np.float32)
        img, m, dep, K = _resize_to_train(img, m, dep, K, train_w, train_h, scale_k=True)
        images.append(img); masks.append(m); depths.append(dep)
        Ks.append(K); c2ws.append(c2w); stems.append(f.stem)

    if da3 is not None:
        print(f"Loaded DA3 poses/depth for {sum(d is not None for d in depths)}/{len(files)} views")
    print(f"Train resolution: {train_w}x{train_h} | views: {len(images)}")
    return {"images": images, "masks": masks, "depths": depths,
            "Ks": np.stack(Ks), "c2ws": np.stack(c2ws),
            "H": train_h, "W": train_w, "n_views": len(images),
            "view_names": stems,
            "scale_mat": np.eye(4, dtype=np.float32)}

img_dir = VIEW_DIR / "images" if (VIEW_DIR / "images").exists() else AERIAL_DIR / IMG_SUBDIR
msk_dir = VIEW_DIR / "masks"  if (VIEW_DIR / "masks").exists()  else AERIAL_DIR / MASK_SUBDIR
_da3 = DA3_NPZ if Path(DA3_NPZ).exists() else None

if DATA_MODE == "neus_bundle" and CAMERAS_NPZ is not None:
    scene = load_neus_bundle(
        CAMERAS_NPZ, img_dir, msk_dir,
        depth_dir=NEUS_DEPTH_DIR,
        meta_npz=SCENE_META_NPZ,
        da3_npz=_da3,
        train_w=TRAIN_W, train_h=TRAIN_H,
    )
    print(f"✓ NeuS cameras + source views: {scene['n_views']} @ {scene['W']}x{scene['H']}")
elif DATA_MODE == "raw_views" or CAMERAS_NPZ is None:
    if not img_dir.exists():
        print(f"⚠ No images at {img_dir} — creating synthetic smoke-test scene")
        # Tiny synthetic sphere scene for offline unit tests
        H = W = 128; n = 4
        images, masks, depths, Ks, c2ws = [], [], [], [], []
        for i in range(n):
            ang = 2 * np.pi * i / n
            c2w = np.eye(4, dtype=np.float32)
            c2w[:3, 3] = [2.5 * np.cos(ang), 0.3, 2.5 * np.sin(ang)]
            # look-at origin
            forward = -c2w[:3, 3]; forward /= np.linalg.norm(forward)
            right = np.cross(np.array([0.,1.,0.]), forward); right /= (np.linalg.norm(right)+1e-8)
            up = np.cross(forward, right)
            c2w[:3, 0] = right; c2w[:3, 1] = up; c2w[:3, 2] = forward
            img = np.full((H, W, 3), 0.7, np.float32)
            yy, xx = np.mgrid[0:H, 0:W]
            m = (((xx - W/2)**2 + (yy - H/2)**2) < (W*0.25)**2).astype(np.float32)
            img[m > 0] = [0.85, 0.55, 0.35]
            images.append(img); masks.append(m); depths.append(None)
            Ks.append(np.array([[W,0,W/2],[0,W,H/2],[0,0,1]], np.float32))
            c2ws.append(c2w)
        scene = {"images": images, "masks": masks, "depths": depths,
                 "Ks": np.stack(Ks), "c2ws": np.stack(c2ws),
                 "H": H, "W": W, "n_views": n, "scale_mat": np.eye(4, np.float32)}
        print(f"✓ Synthetic smoke scene: {n} views @ {W}x{H}")
    else:
        scene = load_raw_views(img_dir, msk_dir, _da3, train_w=TRAIN_W, train_h=TRAIN_H)
        print(f"✓ Raw views: {scene['n_views']} @ {scene['W']}x{scene['H']}")
else:
    raise FileNotFoundError(
        f"neus_bundle mode needs cameras_sphere.npz (run sam_baneus §12) "
        f"and images under {img_dir}"
    )


In [ ]:
# 3.1 — Preview a few views + masks
import matplotlib.pyplot as plt

n_show = min(4, scene["n_views"])
fig, axes = plt.subplots(2, n_show, figsize=(3.2 * n_show, 6))
if n_show == 1:
    axes = np.array([[axes[0]], [axes[1]]])
for i in range(n_show):
    axes[0, i].imshow(scene["images"][i]); axes[0, i].set_title(f"view {i}"); axes[0, i].axis("off")
    axes[1, i].imshow(scene["masks"][i], cmap="gray"); axes[1, i].set_title("mask"); axes[1, i].axis("off")
plt.tight_layout(); plt.show()
print("c2w centers:", scene["c2ws"][:, :3, 3].round(3))


## 3.5 — Stage-1 occupancy teacher (NeuS-norm frame)

Loads `stage1_occupancy.npz` + `baneus_scene_meta.npz` from a prior `sam_baneus` run,
maps the 64³ canonical occupancy into the NeuS unit-sphere frame, and exposes:

- `sample_prob(x_norm)` / `pseudo_sdf_from_prob(p)` — continuous SDF teacher
- `OCC_GRID_N11` — dense occupancy on $[-1,1]^3$ for flow / prefit
- `STAGE1_COORDS` / `STAGE1_ALIGN` — same sparse coords + Sim3 used by Stage-2 densify


In [ ]:
# 3.5 — Load Stage-1 occupancy into NeuS-normalized frame
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
import shutil

# Restore stage1 from Drive if needed
if STAGE1_OCC_NPZ is None and BANEUS_META_DIR is not None and Path(BANEUS_META_DIR).exists():
    _src = BANEUS_META_DIR / "stage1"
    _dst = OUTPUT_DIR / "stage1"
    if _src.is_dir():
        _dst.mkdir(parents=True, exist_ok=True)
        for f in _src.iterdir():
            if f.is_file() and not (_dst / f.name).exists():
                shutil.copy2(f, _dst / f.name)
                print("restored", f.name)
    STAGE1_OCC_NPZ = _first_existing(_dst / "stage1_occupancy.npz", _src / "stage1_occupancy.npz")
    STAGE1_FRAME_NPZ = _first_existing(_dst / "stage1_frame.npz", _src / "stage1_frame.npz")

HAS_STAGE1 = STAGE1_OCC_NPZ is not None and Path(STAGE1_OCC_NPZ).exists()
OCC_GRID_N11 = None
STAGE1_COORDS = None  # exported sparse coords (N,3|4) in 64³ lattice; used by Stage-2
STAGE1_ALIGN = None   # Sim3 + NeuS sphere params shared with Stage-2 densify

if not HAS_STAGE1:
    print("⚠ No stage1_occupancy.npz — prefit/occupancy prior disabled. Run sam_baneus §9 first.")

    def sample_prob(x_norm):
        return torch.zeros(x_norm.shape[0], device=x_norm.device)

    def pseudo_sdf_from_prob(p):
        return p * 0.0

    def canonical_to_neus_norm(x_can):
        raise RuntimeError("no Stage-1 alignment — run §3.5 with stage1_occupancy + baneus_scene_meta")
else:
    occ = np.load(STAGE1_OCC_NPZ)
    prob_can = torch.from_numpy(np.asarray(occ["prob"], np.float32))  # (64,64,64) canonical
    if "coords" in occ.files:
        STAGE1_COORDS = torch.from_numpy(np.asarray(occ["coords"], np.int32))  # spconv needs int32
        print(f"✓ exported Stage-1 coords {tuple(STAGE1_COORDS.shape)} (reuse for Stage-2)")
    else:
        print("⚠ stage1_occupancy.npz has no 'coords' — Stage-2 will re-run sparse structure")
    if SCENE_META_NPZ is None:
        raise FileNotFoundError("baneus_scene_meta.npz required to map Stage-1 → NeuS norm")
    meta = np.load(SCENE_META_NPZ, allow_pickle=True)
    center = torch.tensor(np.asarray(meta["center"], np.float32))
    radius = float(meta["radius"])
    _M = np.asarray(
        meta["M_canonical_to_da3_world"] if "M_canonical_to_da3_world" in meta.files else np.eye(4),
        np.float32,
    )
    _M_inv = torch.tensor(np.linalg.inv(_M), dtype=torch.float32)
    _M_fwd = torch.tensor(_M, dtype=torch.float32)
    _was = float(meta["world_align_s"] if "world_align_s" in meta.files else 1.0)
    _waR = torch.tensor(
        np.asarray(meta["world_align_R"] if "world_align_R" in meta.files else np.eye(3), np.float32)
    )
    _wat = torch.tensor(
        np.asarray(meta["world_align_t"] if "world_align_t" in meta.files else np.zeros(3), np.float32)
    )
    _prob = prob_can
    STAGE1_ALIGN = {
        "M_canonical_to_da3_world": _M_fwd,
        "world_align_s": _was,
        "world_align_R": _waR,
        "world_align_t": _wat,
        "center": center,
        "radius": radius,
        "voxel_grid": 64,
    }

    def neus_norm_to_canonical(x_norm):
        dev = x_norm.device
        x_world = x_norm * radius + center.to(dev)
        x_world = (x_world - _wat.to(dev)) @ _waR.to(dev).T / _was
        ones = torch.ones(x_world.shape[0], 1, device=dev, dtype=x_norm.dtype)
        return (torch.cat([x_world, ones], dim=-1) @ _M_inv.to(dev))[:, :3]

    def canonical_to_neus_norm(x_can):
        """Stage-1 canonical [-0.5,0.5]^3 → NeuS unit-sphere (inverse of neus_norm_to_canonical)."""
        dev = x_can.device
        dtype = x_can.dtype
        ones = torch.ones(x_can.shape[0], 1, device=dev, dtype=dtype)
        x_unal = (torch.cat([x_can, ones], dim=-1) @ _M_fwd.to(device=dev, dtype=dtype))[:, :3]
        x_world = float(_was) * (x_unal @ _waR.to(device=dev, dtype=dtype)) + _wat.to(device=dev, dtype=dtype)
        return (x_world - center.to(device=dev, dtype=dtype)) / float(radius)

    @torch.no_grad()
    def sample_prob(x_norm):
        g = (neus_norm_to_canonical(x_norm) / 0.5).clamp(-0.99, 0.99)
        g = g[:, [2, 1, 0]].view(1, -1, 1, 1, 3)
        v = F.grid_sample(
            _prob.to(x_norm.device)[None, None], g, align_corners=True, padding_mode="border"
        )
        return v.view(-1)

    def pseudo_sdf_from_prob(p):
        return (0.5 - p) * 0.2

    _res = 64
    _lin = torch.linspace(-1, 1, _res)
    _zz, _yy, _xx = torch.meshgrid(_lin, _lin, _lin, indexing="ij")
    _pts = torch.stack([_xx, _yy, _zz], dim=-1).reshape(-1, 3)
    _vals = []
    _bs = 8192
    for i in range(0, _pts.shape[0], _bs):
        _vals.append(sample_prob(_pts[i:i + _bs]))
    OCC_GRID_N11 = torch.cat(_vals).reshape(_res, _res, _res)
    print(
        f"✓ Stage-1 occupancy → NeuS grid {_res}³ | occ mean={float(OCC_GRID_N11.mean()):.3f} "
        f"| frac>0.5={float((OCC_GRID_N11 > 0.5).float().mean()):.3f}"
    )
    print(f"  center={center.numpy().round(3)} radius={radius:.3f}")
    # Round-trip sanity: voxel centers → NeuS → back should land near export
    if STAGE1_COORDS is not None:
        _c = STAGE1_COORDS
        _xyz = _c[:, 1:4].float() if _c.shape[-1] == 4 else _c[:, :3].float()
        _xcan = _xyz / 64.0 - 0.5
        _xn = canonical_to_neus_norm(_xcan)
        _back = neus_norm_to_canonical(_xn)
        _err = (_back - _xcan).norm(dim=-1).mean().item()
        print(f"  Sim3 round-trip mean err={_err:.2e} (should be ~1e-6)")
        print(f"  exported voxels in NeuS: mean={_xn.mean(0).numpy().round(3)} "
              f"std={_xn.std(0).numpy().round(3)} "
              f"|in|<1 frac={float((_xn.abs() < 1).all(-1).float().mean()):.3f}")


### 3.5b — Alignment diagnostics (visual + quantitative)

**Run after §3.5.** Local unit tests live in `tests/test_gfhashba_alignment.py`
(`python tests/test_gfhashba_alignment.py`).

This cell checks:
1. Sim3 round-trip on exported Stage-1 coords
2. Voxel → NeuS soft-splat vs `OCC_GRID_N11` (IoU / corr) — must **not** look like AABB fill
3. Mid-slice overlays (save `align_diag_s1.png`)

In [ ]:
# 3.5b — Alignment diagnostics (quant + visual)
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

# Prefer shared module (upload gfhashba_align.py next to the notebook / on PYTHONPATH)
_repo_candidates = [
    Path("/content/GSV-MSV"),
    Path("/content"),
    Path.cwd(),
    Path("/Users/taiganishida/Documents/GSV-MSV"),
]
for _r in _repo_candidates:
    if (_r / "gfhashba_align.py").exists() and str(_r) not in sys.path:
        sys.path.insert(0, str(_r))

try:
    from gfhashba_align import (
        AlignSim3,
        aabb_stretch_to_grid,
        grid_corr,
        grid_iou,
        occ_bbox_from_grid,
        sim3_soft_splat_to_grid,
        voxel_idx_to_neus,
    )
    _HAS_ALIGN_LIB = True
except Exception as _e:
    _HAS_ALIGN_LIB = False
    # Colab: upload gfhashba_align.py or place it on Drive
    import shutil
    for _cand in (
        Path("/content/drive/MyDrive/SatelliteFuse/GFhashBA/gfhashba_align.py"),
        Path("/content/drive/MyDrive/SatelliteFuse/gfhashba_align.py"),
        Path("/content/GSV-MSV/gfhashba_align.py"),
    ):
        if _cand.exists():
            shutil.copy2(_cand, Path("/content/gfhashba_align.py"))
            if "/content" not in sys.path:
                sys.path.insert(0, "/content")
            try:
                from gfhashba_align import (
                    AlignSim3, aabb_stretch_to_grid, grid_corr, grid_iou,
                    occ_bbox_from_grid, sim3_soft_splat_to_grid, voxel_idx_to_neus,
                )
                _HAS_ALIGN_LIB = True
                print("✓ loaded gfhashba_align from", _cand)
                break
            except Exception as _e2:
                print("still failing after copy:", _e2)
    if not _HAS_ALIGN_LIB:
        print("⚠ gfhashba_align not importable — using notebook-local fallbacks:", _e)
        print("  Upload gfhashba_align.py to /content/ for full AABB-negative tests.")

assert HAS_STAGE1 and STAGE1_ALIGN is not None and OCC_GRID_N11 is not None, "Run §3.5 first"
assert STAGE1_COORDS is not None, "stage1_occupancy.npz needs 'coords' for Stage-2 align tests"

# ---- 1) Sim3 round-trip ----
_c = STAGE1_COORDS
_xyz = _c[:, 1:4].float() if _c.shape[-1] == 4 else _c[:, :3].float()
_xcan = _xyz / float(STAGE1_ALIGN["voxel_grid"]) - 0.5
_xn = canonical_to_neus_norm(_xcan)
_back = neus_norm_to_canonical(_xn)
_rt_err = float((_back - _xcan).norm(dim=-1).mean())
_in_frac = float((_xn.abs() < 1.0).all(dim=-1).float().mean())
print(f"[1] Sim3 round-trip mean err={_rt_err:.2e}  | voxels |x|<1 frac={_in_frac:.3f}")
assert _rt_err < 1e-5, f"Sim3 round-trip broken: {_rt_err}"
assert _in_frac > 0.5, f"Too many Stage-1 voxels outside NeuS ball ({_in_frac:.3f})"

# ---- 2) densify vs OCC (Sim3) + negative AABB check ----
_occ = OCC_GRID_N11.detach().float().cpu().numpy()
_coords_np = _c.detach().cpu().numpy()

if _HAS_ALIGN_LIB:
    _align = AlignSim3.from_mapping(STAGE1_ALIGN)
    _dens = sim3_soft_splat_to_grid(_coords_np, None, _align, res=_occ.shape[0])
    # AABB stretch of the same coords (should score worse vs OCC)
    _aabb_idx = aabb_stretch_to_grid(_coords_np, res=_occ.shape[0])
    _aabb = np.zeros_like(_occ)
    _aabb[_aabb_idx[:, 2], _aabb_idx[:, 1], _aabb_idx[:, 0]] = 1.0
else:
    # fallback: splat via notebook canonical_to_neus_norm
    _res = _occ.shape[0]
    _dens = np.zeros((_res, _res, _res), np.float64)
    _g = ((_xn.cpu().numpy() + 1.0) * 0.5 * (_res - 1))
    for p in _g:
        ix, iy, iz = [int(np.clip(round(v), 0, _res - 1)) for v in p]
        _dens[iz, iy, ix] += 1.0
    if _dens.max() > 0:
        _dens /= _dens.max()
    _aabb = None

_iou = float(((_dens > 0.1) & (_occ > 0.5)).sum() / max(((_dens > 0.1) | (_occ > 0.5)).sum(), 1))
# pearson
_dn = (_dens - _dens.mean()) / (_dens.std() + 1e-8)
_on = (_occ - _occ.mean()) / (_occ.std() + 1e-8)
_corr = float((_dn * _on).mean())
print(f"[2] Sim3 densify vs OCC_GRID_N11: IoU={_iou:.3f}  corr={_corr:.3f}")
if _iou < 0.05:
    raise RuntimeError(
        f"Align FAIL: IoU={_iou:.3f} — Stage-1 coords do not land on OCC in NeuS. "
        "Check baneus_scene_meta / pinned SamBAneus run."
    )
if _aabb is not None:
    _iou_aabb = float(((_aabb > 0.5) & (_occ > 0.5)).sum() / max(((_aabb > 0.5) | (_occ > 0.5)).sum(), 1))
    print(f"    AABB-stretch vs OCC (should be worse): IoU={_iou_aabb:.3f}")
    if _iou_aabb > _iou + 0.05:
        print("⚠ AABB scored higher than Sim3 — suspicious; inspect overlays")

# ---- 3) visual mid-slices ----
_mid = _occ.shape[0] // 2
fig, axs = plt.subplots(2, 3, figsize=(11, 7))
for ax, img, title in (
    (axs[0, 0], _dens[_mid], "Sim3 dens (z mid)"),
    (axs[0, 1], _occ[_mid], "OCC NeuS (z mid)"),
    (axs[0, 2], _dens[_mid] * _occ[_mid], "product"),
    (axs[1, 0], _dens[:, _mid, :], "Sim3 dens (y mid)"),
    (axs[1, 1], _occ[:, _mid, :], "OCC NeuS (y mid)"),
    (axs[1, 2], (_aabb[_mid] if _aabb is not None else np.zeros_like(_occ[_mid])), "AABB dens (z mid) BAD"),
):
    ax.imshow(img, origin="lower", cmap="magma")
    ax.set_title(title)
    ax.axis("off")
fig.suptitle(f"Align diag | IoU={_iou:.3f} corr={_corr:.3f} | rt_err={_rt_err:.1e}")
fig.tight_layout()
_png = OUTPUT_DIR / "align_diag_s1.png"
fig.savefig(_png, dpi=130, bbox_inches="tight")
plt.show()
print("✓ saved", _png)
print("PASS §3.5b alignment gates" if _iou >= 0.05 and _rt_err < 1e-5 else "FAIL")

# ---- paste-friendly summary for chat debug ----
_snap = {
    "section": "3.5b",
    "sim3_roundtrip_err": _rt_err,
    "voxels_in_neus_frac": _in_frac,
    "densify_vs_occ_iou": _iou,
    "densify_vs_occ_corr": _corr,
    "n_stage1_coords": int(_coords_np.shape[0]),
    "occ_mean": float(_occ.mean()),
    "occ_frac_gt_0.5": float((_occ > 0.5).mean()),
    "neus_center": [float(x) for x in (STAGE1_ALIGN["center"].detach().cpu().numpy().ravel() if hasattr(STAGE1_ALIGN["center"], "detach") else np.asarray(STAGE1_ALIGN["center"]).ravel())],
    "neus_radius": float(STAGE1_ALIGN["radius"]),
    "FLOW_PRIOR_ON_VISIBLE": bool(globals().get("FLOW_PRIOR_ON_VISIBLE", False)),
    "W_FLOW_START": float(globals().get("W_FLOW_START", -1)),
    "FLOW_HOLD_FRAC": float(globals().get("FLOW_HOLD_FRAC", -1)),
    "SAM_BANEU_RUN": str(globals().get("SAM_BANEU_RUN")),
    "RUN_TS": str(globals().get("RUN_TS")),
}
import json as _json
_snap_path = OUTPUT_DIR / "debug_snapshot_3_5b.json"
_snap_path.write_text(_json.dumps(_snap, indent=2))
print("=== DEBUG SNAPSHOT §3.5b (paste this block) ===")
print(_json.dumps(_snap, indent=2))
print("=== END SNAPSHOT ===")
print("also wrote", _snap_path)

## 4 — Core modules (the four components)

Implemented as pure PyTorch so the notebook is self-contained. If `tinycudann`
is installed, `HashGridEncoding` will prefer it; otherwise we use a compatible
multi-resolution hash fallback.


### 4.1 — Multi-resolution Hash encoding + Coarse-to-Fine Annealing (component 4)

Hash grids learn high frequency immediately — fatal for early pose BA.
We introduce a continuous capacity $\gamma(k)\in[1,L]$ and weight level $l$:

$$
W_l(k)=\tfrac12\Bigl(1-\cos\bigl(\pi\cdot\mathrm{clamp}(\gamma(k)-l,\,0,\,1)\bigr)\Bigr)
$$

$\gamma$ ramps $1\to L$ over the first `ANNEAL_ITERS` steps, so the Hash-SDF
behaves like a low-frequency MLP until cameras lock.


In [ ]:
# 4.1 — Hash grid encoding + annealing
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import tinycudann as tcnn
    HAS_TCNN = True
except Exception:
    HAS_TCNN = False
print(f"tinycudann available: {HAS_TCNN}")


def hash_anneal_weights(n_levels: int, gamma: float, device=None, dtype=torch.float32):
    """W_l = 0.5 * (1 - cos(pi * clamp(gamma - l, 0, 1))) for l = 0..L-1."""
    levels = torch.arange(n_levels, device=device, dtype=dtype)
    t = (gamma - levels).clamp(0.0, 1.0)
    return 0.5 * (1.0 - torch.cos(math.pi * t))


def anneal_gamma(iteration: int, n_levels: int, anneal_iters: int, start: float = 1.0) -> float:
    if anneal_iters <= 0:
        return float(n_levels)
    t = min(max(iteration / float(anneal_iters), 0.0), 1.0)
    return float(start + t * (n_levels - start))


class HashGridEncoding(nn.Module):
    """Multi-res hash grid. TCNN if available, else pure-PyTorch fallback."""

    def __init__(
        self,
        n_levels=16,
        n_features_per_level=2,
        log2_hashmap_size=19,
        base_resolution=16,
        max_resolution=2048,
        dtype=torch.float32,
    ):
        super().__init__()
        self.n_levels = n_levels
        self.n_features = n_features_per_level
        self.out_dim = n_levels * n_features_per_level
        self.base_resolution = base_resolution
        self.max_resolution = max_resolution
        self.log2_hashmap_size = log2_hashmap_size
        b = math.exp((math.log(max_resolution) - math.log(base_resolution)) / max(n_levels - 1, 1))
        self.register_buffer("level_scale", torch.tensor(b, dtype=dtype))

        if HAS_TCNN:
            cfg = {
                "otype": "HashGrid",
                "n_levels": n_levels,
                "n_features_per_level": n_features_per_level,
                "log2_hashmap_size": log2_hashmap_size,
                "base_resolution": base_resolution,
                "per_level_scale": b,
            }
            self.enc = tcnn.Encoding(3, cfg)
            self.backend = "tcnn"
        else:
            hashmap_size = 1 << log2_hashmap_size
            self.hashmap_size = hashmap_size
            # One table per level: [hashmap_size, n_features]
            self.tables = nn.ParameterList([
                nn.Parameter(torch.empty(hashmap_size, n_features_per_level, dtype=dtype))
                for _ in range(n_levels)
            ])
            for p in self.tables:
                nn.init.uniform_(p, -1e-4, 1e-4)
            self.register_buffer(
                "primes",
                torch.tensor([1, 2654435761, 805459861], dtype=torch.int64),
            )
            self.backend = "torch"

    def _hash(self, x_int):
        # x_int: (..., 3) int64
        xh = (x_int * self.primes).bitwise_xor(x_int.new_zeros(x_int.shape[:-1]).unsqueeze(-1))
        # fold 3 coords
        h = x_int[..., 0] * self.primes[0]
        h = torch.bitwise_xor(h, x_int[..., 1] * self.primes[1])
        h = torch.bitwise_xor(h, x_int[..., 2] * self.primes[2])
        return torch.remainder(h, self.hashmap_size)

    def forward(self, x, gamma: float | None = None):
        """
        x: (N, 3) in roughly [-1, 1] (we map to [0, 1] for hashing).
        gamma: if set, apply coarse-to-fine level weights.
        """
        if self.backend == "tcnn":
            # tcnn expects [0,1]
            x01 = (x.clamp(-1, 1) + 1.0) * 0.5
            feat = self.enc(x01.float()).to(dtype=x.dtype)
            feat = feat.view(-1, self.n_levels, self.n_features)
        else:
            x01 = (x.clamp(-1, 1) + 1.0) * 0.5
            feats = []
            for lvl in range(self.n_levels):
                res = self.base_resolution * (self.level_scale ** lvl)
                x_scaled = x01 * res
                x0 = x_scaled.floor().long()
                w = x_scaled - x0.float()
                # 8 corners
                corners = []
                weights = []
                for dz in (0, 1):
                    for dy in (0, 1):
                        for dx in (0, 1):
                            c = x0 + torch.tensor([dx, dy, dz], device=x.device, dtype=torch.long)
                            corners.append(self._hash(c))
                            ww = ((w[..., 0] if dx else (1 - w[..., 0])) *
                                  (w[..., 1] if dy else (1 - w[..., 1])) *
                                  (w[..., 2] if dz else (1 - w[..., 2])))
                            weights.append(ww)
                table = self.tables[lvl]
                acc = 0.0
                for h, ww in zip(corners, weights):
                    acc = acc + table[h] * ww.unsqueeze(-1)
                feats.append(acc)
            feat = torch.stack(feats, dim=1)  # (N, L, F)

        if gamma is not None:
            W = hash_anneal_weights(self.n_levels, gamma, device=feat.device, dtype=feat.dtype)
            feat = feat * W.view(1, -1, 1)
        return feat.reshape(feat.shape[0], -1)


# quick anneal curve check
_g = [anneal_gamma(i, 16, 5000) for i in (0, 1000, 2500, 5000, 10000)]
print("gamma@iters", _g)
print("W@γ=3.5:", hash_anneal_weights(8, 3.5).numpy().round(3))


def flow_prior_schedule(
    iteration: int,
    anneal_iters: int | None = None,
    w_start: float | None = None,
    w_end: float | None = None,
    hold_frac: float | None = None,
):
    """Stage-2 loss weight over main train (after Stage-1 prefit).

    Design (vs photo W_PHOTO=1 every iter; flow only on adaptive cadence):
      1) HOLD W_FLOW_START for FLOW_HOLD_FRAC — strong early prior as basin-finder /
         smoother while the photometric surface is still ill-behaved.
      2) COSINE decay to W_FLOW_END — hand off to photo once near a good minimum.
      3) Floor W_FLOW_END > 0 — small residual on unseen regions late.

    Start ~0.7 (not ~0.35): early goal is leading into the basin, not staying shy
    of photo. Dense early cadence (~/12) ⇒ amortized rate ≈ W/period ~0.06 early,
    then tiny late. gate always 1.0 here; spatial override = FLOW_PRIOR_ON_VISIBLE.
    """
    import math as _math
    A = int(anneal_iters if anneal_iters is not None else FLOW_ANNEAL_ITERS)
    ws = float(w_start if w_start is not None else W_FLOW_START)
    we = float(w_end if w_end is not None else W_FLOW_END)
    hf = float(hold_frac if hold_frac is not None else FLOW_HOLD_FRAC)
    hf = min(max(hf, 0.0), 0.95)
    hold_end = int(hf * max(1, A))

    if iteration <= hold_end:
        w_flow = ws
    else:
        # cosine from ws → we over (hold_end, A]
        span = max(1, A - hold_end)
        t = min(1.0, float(iteration - hold_end) / span)
        w_flow = we + 0.5 * (ws - we) * (1.0 + _math.cos(_math.pi * t))

    gate = 1.0
    return float(w_flow), float(gate)



def flow_call_period(iteration: int) -> int:
    """How often to run Stage-2 distill (adaptive).

    Dense while W_flow is held high (geometry plastic / basin-finding), then
    sparsify so late training is mostly photometric without transformer cost
    every step. Aligned to FLOW_HOLD_FRAC ≈ 0.35.
    """
    A = max(1, int(FLOW_ANNEAL_ITERS))
    t = min(1.0, float(iteration) / A)
    if t < 0.35:
        return 12    # hold phase: frequent strong kicks
    if t < 0.55:
        return 25    # early decay
    if t < 0.80:
        return 50
    return 100       # late: rare residual


def flow_proxy_res(iteration: int) -> int:
    """Slightly coarser proxy early (when we call often); full PROXY_RES later."""
    A = max(1, int(FLOW_ANNEAL_ITERS))
    if iteration < float(FLOW_HOLD_FRAC) * A:
        return min(32, int(PROXY_RES))
    return int(PROXY_RES)


def plot_flow_prior_schedule(save_path=None, end_iter=None, anneal_iters=None):
    """Graph hold-then-cosine W_flow (and schedule gate) vs iteration."""
    import matplotlib.pyplot as plt
    import numpy as np
    E = int(end_iter if end_iter is not None else END_ITER)
    A = int(anneal_iters if anneal_iters is not None else FLOW_ANNEAL_ITERS)
    xs = np.arange(0, E)
    ws, gs = [], []
    for it in xs:
        w, g = flow_prior_schedule(int(it), anneal_iters=A)
        ws.append(w); gs.append(g)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(xs, ws, lw=2, label=r"$W_{\mathrm{flow}}(t)$ (hold → cosine)")
    ax.plot(xs, gs, lw=2, label=r"schedule gate $g$ (1=visibility-gated)")
    hold_it = int(float(FLOW_HOLD_FRAC) * A)
    ax.axvline(hold_it, color="C2", ls=":", lw=1.5, label=f"hold end ({FLOW_HOLD_FRAC:.0%})")
    ax.axvline(A, color="gray", ls="--", lw=1, label=f"FLOW_ANNEAL_ITERS={A}")
    ax.set_xlabel("iteration")
    ax.set_ylabel("weight / gate")
    ax.set_title("Flow prior: strong early hold → cosine handoff to photo")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    if save_path is None:
        save_path = OUTPUT_DIR / "flow_prior_schedule.png"
    fig.savefig(save_path, dpi=140, bbox_inches="tight")
    plt.show()
    print("Saved", save_path)
    return save_path


### 4.2 — Hash-SDF + Background NeRF + Scene Contraction (component 3)

Foreground building lives in normalized $[-1,1]^3$. Outside is warped:

$$
M_{\mathrm{contract}}(\mathbf{x})=\begin{cases}
\mathbf{x} & \|\mathbf{x}\|\le 1\\
\bigl(2-\frac{1}{\|\mathbf{x}\|}\bigr)\frac{\mathbf{x}}{\|\mathbf{x}\|} & \|\mathbf{x}\|>1
\end{cases}
$$

Mask BCE forces foreground opacity $O_{\mathrm{fg}}$ to match SAM masks so the
background field owns sky/trees.


In [ ]:
# 4.2 — Contraction, Hash-SDF, background NeRF, color head
import torch
import torch.nn as nn
import torch.nn.functional as F


def contract(x, ord_p: str = "inf"):
    """Mip-NeRF 360-style contraction. x: (N, 3)."""
    if ord_p == "inf":
        mag = x.abs().amax(dim=-1, keepdim=True)
    else:
        mag = torch.linalg.norm(x, dim=-1, keepdim=True)
    inside = mag <= 1.0
    # (2 - 1/||x||) * x/||x||
    x_out = (2.0 - 1.0 / mag.clamp_min(1e-6)) * (x / mag.clamp_min(1e-6))
    return torch.where(inside, x, x_out)


def sdf_to_density(sdf, alpha, beta):
    """σ(x) = α · sigmoid(-β · f(x))  — NeuS / VolSDF-style bridge."""
    return alpha * torch.sigmoid(-beta * sdf)


class HashSDF(nn.Module):
    def __init__(self, encoding: HashGridEncoding, hidden=64, n_layers=2, geo_init_bias=0.5):
        super().__init__()
        self.encoding = encoding
        layers = []
        in_d = encoding.out_dim
        for i in range(n_layers):
            layers += [nn.Linear(in_d if i == 0 else hidden, hidden), nn.ReLU(inplace=True)]
        self.mlp = nn.Sequential(*layers)
        self.sdf_out = nn.Linear(hidden, 1)
        self.feat_out = nn.Linear(hidden, hidden)
        # geometric init: sphere-ish bias
        nn.init.normal_(self.sdf_out.weight, 0, 1e-4)
        nn.init.constant_(self.sdf_out.bias, geo_init_bias)
        # learnable Laplace/sigmoid sharpness for density conversion
        self.log_beta = nn.Parameter(torch.tensor(1.6094))   # β≈5
        self.log_alpha = nn.Parameter(torch.tensor(0.0))     # α≈1

    @property
    def beta(self):
        return self.log_beta.exp().clamp(0.1, 50.0)

    @property
    def alpha(self):
        return self.log_alpha.exp().clamp(0.1, 50.0)

    def forward(self, x, gamma=None):
        h = self.encoding(x, gamma=gamma)
        h = self.mlp(h)
        sdf = self.sdf_out(h)
        feat = self.feat_out(h)
        return sdf, feat

    def density(self, x, gamma=None):
        sdf, feat = self.forward(x, gamma=gamma)
        return sdf_to_density(sdf, self.alpha, self.beta), sdf, feat


class BackgroundNeRF(nn.Module):
    """Tiny MLP on contracted coords for sky / distant clutter."""

    def __init__(self, hidden=64, n_layers=4, n_freq=6):
        super().__init__()
        self.n_freq = n_freq
        in_d = 3 + 3 * 2 * n_freq
        layers = [nn.Linear(in_d, hidden), nn.ReLU(inplace=True)]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.ReLU(inplace=True)]
        self.mlp = nn.Sequential(*layers)
        self.sigma = nn.Linear(hidden, 1)
        self.rgb = nn.Linear(hidden, 3)

    def pe(self, x):
        out = [x]
        for i in range(self.n_freq):
            out += [torch.sin((2 ** i) * math.pi * x), torch.cos((2 ** i) * math.pi * x)]
        return torch.cat(out, dim=-1)

    def forward(self, x):
        xc = contract(x)
        h = self.mlp(self.pe(xc))
        return F.relu(self.sigma(h)), torch.sigmoid(self.rgb(h))


class ColorNet(nn.Module):
    def __init__(self, feat_dim=64, hidden=64, n_freq=4):
        super().__init__()
        in_d = feat_dim + 3 + 3 + 3 + 3 * 2 * n_freq  # feat + x + normal + dir + dir PE
        self.n_freq = n_freq
        self.net = nn.Sequential(
            nn.Linear(in_d, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, 3),
        )

    def pe(self, d):
        out = [d]
        for i in range(self.n_freq):
            out += [torch.sin((2 ** i) * math.pi * d), torch.cos((2 ** i) * math.pi * d)]
        return torch.cat(out, dim=-1)

    def forward(self, x, n, d, feat):
        inp = torch.cat([feat, x, n, self.pe(d)], dim=-1)
        return torch.sigmoid(self.net(inp))


print("✓ HashSDF / BackgroundNeRF / ColorNet defined")


### 4.3 — Continuous Generative Score Guidance (component 1)

SDF → density proxy $z_0$ → distill a frozen generative velocity.

**Default:** `FLOW_PRIOR_MODE="stage2"` — real MV-SAM3D Stage-2 `reverse_fn` velocity (§6.0a), Sim3-mapped + soft-splatted into NeuS $[-1,1]^3$ (same frame as Stage-1 prefit).

Fallbacks: `"occupancy"` (Stage-1 grid only), `"external"` + `attach()`, `"off"`.


In [ ]:
# 4.3 — SDF→density proxy + flow / occupancy distillation
import torch
import torch.nn as nn
import torch.nn.functional as F


class StubFlowPrior(nn.Module):
    """DEV ONLY — ellipsoid blob teacher. Do not use for real runs."""

    def __init__(self, target_blob_sigma=0.35):
        super().__init__()
        self.target_blob_sigma = target_blob_sigma

    def make_target(self, res, device, dtype):
        lin = torch.linspace(-1, 1, res, device=device, dtype=dtype)
        zz, yy, xx = torch.meshgrid(lin, lin, lin, indexing="ij")
        r2 = (xx / 0.7) ** 2 + (yy / 0.9) ** 2 + (zz / 0.7) ** 2
        return torch.exp(-0.5 * r2 / (self.target_blob_sigma ** 2))

    @torch.no_grad()
    def velocity(self, z_t, t, cond=None, z0_shape=None):
        res = (z0_shape or z_t.shape)[-1]
        target = self.make_target(res, z_t.device, z_t.dtype)
        while target.ndim < z_t.ndim:
            target = target.unsqueeze(0)
        return target - z_t


class OccupancyFlowPrior(nn.Module):
    """Stage-1 occupancy teacher: velocity pulls proxy density toward building-shaped grid."""

    def __init__(self, occ_prob_n11: torch.Tensor):
        super().__init__()
        assert occ_prob_n11.ndim == 3
        self.register_buffer("target", occ_prob_n11.float(), persistent=True)

    @torch.no_grad()
    def velocity(self, z_t, t, cond=None, z0_shape=None):
        tgt = self.target
        if tgt.shape[-1] != z_t.shape[-1]:
            tgt = F.interpolate(
                tgt[None, None], size=z_t.shape[-3:], mode="trilinear", align_corners=True
            )[0, 0]
        while tgt.ndim < z_t.ndim:
            tgt = tgt.unsqueeze(0)
        return tgt.to(dtype=z_t.dtype) - z_t



class Stage2FlowPrior(nn.Module):
    """Real MV-SAM3D Stage-2 (SLAT Flow Transformer) velocity → dense NeuS prior.

    Stage-2 lives in *sparse SLAT* space (N×8 + coords + image cond).
    GF-HashBA distills a *dense* density proxy on the NeuS unit sphere.
    Bridge (MUST match §3.5 Sim3 — never AABB-stretch):
      1) Run Stage-2 reverse_fn velocity in SLAT at time t
      2) Recover clean SLAT estimate
      3) Map voxel idx → x_can = idx/G − 0.5 → Sim3 → NeuS x_norm
      4) Soft-splat into dense [-1,1]^3; return v = target_dense − z_t
    """

    def __init__(
        self,
        slat_generator,
        cond_args,
        cond_kwargs,
        coords,
        x1_slat,
        align: dict,
        time_scale: float = 1000.0,
        grid_res: int = 64,
        dtype=torch.float32,
    ):
        super().__init__()
        if align is None:
            raise ValueError("Stage2FlowPrior requires STAGE1_ALIGN (run §3.5 first)")
        self.slat_gen = slat_generator
        self.cond_args = cond_args
        self.cond_kwargs = dict(cond_kwargs or {})
        self.time_scale = float(time_scale)
        self.grid_res = int(grid_res)
        self.voxel_grid = int(align.get("voxel_grid", 64))
        if coords.shape[-1] == 4:
            cxyz = coords[:, 1:4]
        else:
            cxyz = coords[:, :3]
        self.register_buffer("coords_xyz", cxyz.to(dtype=torch.int32), persistent=False)
        self.register_buffer("x1_slat", x1_slat.float(), persistent=False)
        self.register_buffer("M_can2w", align["M_canonical_to_da3_world"].float(), persistent=False)
        self.register_buffer("wa_R", align["world_align_R"].float(), persistent=False)
        self.register_buffer("wa_t", align["world_align_t"].float(), persistent=False)
        self.register_buffer("neus_center", align["center"].float(), persistent=False)
        self.wa_s = float(align["world_align_s"])
        self.neus_radius = float(align["radius"])
        if hasattr(self.slat_gen, "reverse_fn") and hasattr(self.slat_gen.reverse_fn, "strength"):
            self.slat_gen.reverse_fn.strength = 0.0
        if hasattr(self.slat_gen, "no_shortcut"):
            self.slat_gen.no_shortcut = True
        self._dtype = dtype

    def coords_to_neus(self, coords_xyz, device=None, dtype=None):
        """Integer SLAT/SS indices → NeuS x_norm via coords/G−0.5 + Sim3."""
        device = device or coords_xyz.device
        dtype = dtype or torch.float32
        c = coords_xyz.to(device=device, dtype=dtype)
        x_can = c / float(self.voxel_grid) - 0.5
        ones = torch.ones(x_can.shape[0], 1, device=device, dtype=dtype)
        x_unal = (torch.cat([x_can, ones], dim=-1) @ self.M_can2w.to(device=device, dtype=dtype))[:, :3]
        x_world = self.wa_s * (x_unal @ self.wa_R.to(device=device, dtype=dtype)) + self.wa_t.to(device=device, dtype=dtype)
        return (x_world - self.neus_center.to(device=device, dtype=dtype)) / self.neus_radius

    def slat_feats_to_dense(self, coords_xyz, feats, res: int, device, dtype):
        """Sim3-aligned soft splat of SLAT energy onto NeuS [-1,1]^3 (vol[z,y,x])."""
        x_norm = self.coords_to_neus(coords_xyz, device=device, dtype=dtype)
        # continuous grid indices matching extract_density_proxy / OCC_GRID_N11 layout
        g = (x_norm + 1.0) * 0.5 * (res - 1)  # (N,3) as (x,y,z)
        inside = (x_norm.abs() <= 1.05).all(dim=-1)
        if feats is None:
            w = torch.ones(g.shape[0], device=device, dtype=dtype)
        else:
            f = feats.to(device=device, dtype=dtype)
            if f.ndim == 3:
                f = f[0]
            w = f.norm(dim=-1)
            w = w / w.mean().clamp_min(1e-4)
        g = g[inside]
        w = w[inside]
        vol = torch.zeros(res, res, res, device=device, dtype=dtype)
        if g.numel() == 0:
            return vol
        x, y, z = g[:, 0], g[:, 1], g[:, 2]
        x0 = x.floor().long(); y0 = y.floor().long(); z0 = z.floor().long()
        for dz, wz in ((0, 1 - (z - z0.float())), (1, (z - z0.float()))):
            for dy, wy in ((0, 1 - (y - y0.float())), (1, (y - y0.float()))):
                for dx, wx in ((0, 1 - (x - x0.float())), (1, (x - x0.float()))):
                    ix = (x0 + dx).clamp(0, res - 1)
                    iy = (y0 + dy).clamp(0, res - 1)
                    iz = (z0 + dz).clamp(0, res - 1)
                    ww = (wz * wy * wx * w).to(dtype)
                    vol.index_put_((iz, iy, ix), ww, accumulate=True)
        vol = vol / vol.max().clamp_min(1e-4)
        # mild blur to avoid sparse voxel "grid sparkles"
        k = torch.ones(1, 1, 3, 3, 3, device=device, dtype=dtype) / 27.0
        vol = F.conv3d(vol[None, None], k, padding=1)[0, 0]
        vol = F.conv3d(vol[None, None], k, padding=1)[0, 0]
        return vol.clamp(0, 1)

    @torch.no_grad()
    def velocity(self, z_t, t, cond=None, z0_shape=None):
        device = z_t.device
        dtype = z_t.dtype
        # FlowMatching.loss passes `t * time_scale` as a TENSOR (not Python float).
        if not torch.is_tensor(t):
            t = torch.tensor(t, device=device, dtype=dtype)
        t_01 = t.reshape(-1)[0].to(device=device, dtype=dtype).clamp(0.02, 0.98)
        t_s = float(t_01.item())
        t_in = (t_01 * float(self.time_scale)).reshape(1)

        x1 = self.x1_slat.to(device=device, dtype=dtype)
        if x1.ndim == 2:
            x1 = x1.unsqueeze(0)
        noise = torch.randn_like(x1)
        x_t = (1.0 - t_s) * noise + t_s * x1
        rev = self.slat_gen.reverse_fn
        cond_args = []
        for a in self.cond_args:
            if torch.is_tensor(a):
                cond_args.append(a.to(device=device))
            else:
                cond_args.append(a)
        cond_args = tuple(cond_args)
        cond_kwargs = {
            k: (v.to(device=device) if torch.is_tensor(v) else v)
            for k, v in self.cond_kwargs.items()
        }
        try:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                v = rev(x_t, t_in, *cond_args, **cond_kwargs)
        except Exception:
            v = rev(x_t, t_in, *cond_args, **cond_kwargs)
        if isinstance(v, (tuple, list)):
            v = v[0]
        if hasattr(v, "feats"):
            v_feats = v.feats
            if v_feats.ndim == 2:
                v_feats = v_feats.unsqueeze(0)
            x1_hat = x_t + (1.0 - t_s) * v_feats
        else:
            x1_hat = x_t + (1.0 - t_s) * v
        res = int((z0_shape or z_t.shape)[-1])
        target = self.slat_feats_to_dense(self.coords_xyz, x1_hat, res, device, dtype)
        while target.ndim < z_t.ndim:
            target = target.unsqueeze(0)
        return target.to(dtype=dtype) - z_t


class ExternalFlowPrior(nn.Module):
    """Hook for Trellis / MV-SAM3D. attach(model) where model(z_t, t, cond) → velocity."""

    def __init__(self, model=None):
        super().__init__()
        self.model = model
        if model is None:
            print("ExternalFlowPrior: call prior.attach(model) before training")

    def attach(self, model):
        self.model = model
        return self

    @torch.no_grad()
    def velocity(self, z_t, t, cond=None, z0_shape=None):
        if self.model is None:
            raise RuntimeError("ExternalFlowPrior has no model — attach() first")
        if callable(self.model) and not isinstance(self.model, nn.Module):
            return self.model(z_t, t, cond)
        return self.model(z_t, t, cond)


def extract_density_proxy(sdf_net: "HashSDF", res: int, gamma, device, dtype=torch.float32):
    lin = torch.linspace(-1, 1, res, device=device, dtype=dtype)
    zz, yy, xx = torch.meshgrid(lin, lin, lin, indexing="ij")
    pts = torch.stack([xx, yy, zz], dim=-1).reshape(-1, 3)
    dens = []
    chunk = max(4096, (res * res * res) // 8)
    for i in range(0, pts.shape[0], chunk):
        d, _, _ = sdf_net.density(pts[i:i + chunk], gamma=gamma)
        dens.append(d)
    return torch.cat(dens, dim=0).reshape(1, 1, res, res, res)


def flow_distillation_loss(
    sdf_net,
    prior: nn.Module,
    res: int = 64,
    gamma=None,
    t_min: float = 0.02,
    t_max: float = 0.98,
    cond=None,
    vis_weight: torch.Tensor | None = None,
    eta_photo: float = 0.0,
    photo_grad_proxy: torch.Tensor | None = None,
):
    """
    Distill prior into z0(θ). Optional ODE-style fusion:
      v_eff = v_θ - η * photo_proxy
    vis_weight (R,R,R): downweight where V high (cameras see it).
    """
    device = next(sdf_net.parameters()).device
    dtype = next(sdf_net.parameters()).dtype
    z0 = extract_density_proxy(sdf_net, res, gamma, device, dtype)

    t = torch.empty(1, device=device, dtype=dtype).uniform_(t_min, t_max)
    eps = torch.randn_like(z0)
    z_t = (1.0 - t) * z0 + t * eps

    with torch.no_grad():
        v_pred = prior.velocity(z_t.detach(), t, cond=cond, z0_shape=z0.shape)
        if photo_grad_proxy is not None and eta_photo > 0:
            g = photo_grad_proxy.to(device=device, dtype=dtype)
            if g.shape != v_pred.shape:
                g = F.interpolate(g, size=v_pred.shape[-3:], mode="trilinear", align_corners=True)
            v_eff = v_pred - float(eta_photo) * g
        else:
            v_eff = v_pred
        z0_target = (z_t - t * v_eff).detach()

    err = (z0 - z0_target) ** 2
    if vis_weight is not None:
        w = (1.0 - vis_weight.detach().clamp(0, 1)).to(dtype=dtype)
        while w.ndim < err.ndim:
            w = w.unsqueeze(0)
        if w.shape[-3:] != err.shape[-3:]:
            w = F.interpolate(
                w if w.ndim == 5 else w[None, None],
                size=err.shape[-3:], mode="trilinear", align_corners=True,
            )
            if w.shape[0] != err.shape[0]:
                w = w.expand_as(err)
        loss = (err * w).sum() / w.sum().clamp_min(1e-4)
    else:
        loss = err.mean()
    return loss, {"z0_mean": float(z0.detach().mean()), "t": float(t.item()), "proxy_res": res}


def build_flow_prior(mode: str = "occupancy", occ_grid_n11: torch.Tensor | None = None, stage2_prior=None):
    mode = (mode or "off").lower()
    if mode in ("off", "none", "false"):
        return None
    if mode == "stub":
        print("⚠ StubFlowPrior — blob teacher (smoke tests only)")
        return StubFlowPrior()
    if mode == "occupancy":
        if occ_grid_n11 is None:
            raise RuntimeError("occupancy prior needs Stage-1 grid — run §3.5 first")
        return OccupancyFlowPrior(occ_grid_n11)
    if mode in ("stage2", "slat", "mvsam3d"):
        if stage2_prior is None:
            raise RuntimeError("stage2 prior: run §6.0a build_stage2_flow_prior() first")
        return stage2_prior
    if mode == "external":
        return ExternalFlowPrior(None)
    raise ValueError(f"Unknown FLOW_PRIOR_MODE={mode}")


print("✓ flow / occupancy priors ready")


### 4.4 — Multi-view visibility $V(\mathbf{x})$ + gated fusion

Physical multi-view visibility at 3D points $\mathbf{x}$:

$$
V(\mathbf{x})=1-\exp\Bigl(-\sum_{i=1}^{N}\omega_i(\mathbf{x})\Bigr)
$$

Each $\omega_i\in\{0,1\}$ is 1 only if the point:
1. projects inside the image and **SAM mask**, and
2. passes a **Z-buffer occlusion** test when depth is available:
   $z_{\mathrm{cam}} \le z_{\mathrm{surf}}(u,v)\,(1+\varepsilon_{\mathrm{rel}})+\varepsilon_{\mathrm{abs}}$.

Silhouette-only projection is *x-ray* (back faces still hit the mask). Depth comes from
DA3 / exported maps already in `scene["depths"]` — an $O(1)$ lookup, not an SDF ray-march.


In [ ]:
# 4.4 — Multi-view visibility V(x) + gated residual fusion
# IMPORTANT: silhouette-only projection is x-ray (back faces land in the mask).
# When per-view depths are available (DA3 / NeuS depth), require
#   z_cam <= z_surface*(1+rel_eps) + abs_eps
# so occluded / back-surface points get ω=0 and the prior stays ON.
import numpy as np
import torch
import torch.nn.functional as F


def rendering_weights_along_ray(density, deltas):
    """density, deltas: (N_rays, N_samp) → weights ω, transmittance T."""
    alpha = 1.0 - torch.exp(-density * deltas)
    T = torch.cumprod(
        torch.cat([torch.ones_like(alpha[:, :1]), 1.0 - alpha[:, :-1] + 1e-7], dim=-1),
        dim=-1,
    )
    return alpha * T, T


def project_points(pts, c2w, K):
    """pts (N,3) in world → u,v,z_cam, in_front. c2w maps cam→world."""
    R = c2w[:3, :3]
    t = c2w[:3, 3]
    Xc = (pts - t[None, :]) @ R  # == R.T @ (pts - t) for row vectors
    z = Xc[:, 2]
    in_front = z > 1e-4
    u = K[0, 0] * (Xc[:, 0] / z.clamp_min(1e-4)) + K[0, 2]
    v = K[1, 1] * (Xc[:, 1] / z.clamp_min(1e-4)) + K[1, 2]
    return u, v, z, in_front


def _stack_depth_maps(depths, H, W, device, dtype):
    """List/array of HxW depths → (V,H,W); missing → zeros (treated as invalid)."""
    if depths is None:
        return None
    if torch.is_tensor(depths):
        d = depths.to(device=device, dtype=dtype)
        if d.ndim == 3:
            return d
        raise ValueError(f"depths tensor must be (V,H,W), got {tuple(d.shape)}")
    maps = []
    for d in depths:
        if d is None:
            maps.append(torch.zeros(H, W, device=device, dtype=dtype))
        else:
            dd = torch.as_tensor(np.asarray(d, np.float32), device=device, dtype=dtype)
            if dd.shape != (H, W):
                dd = F.interpolate(
                    dd[None, None], size=(H, W), mode="nearest"
                )[0, 0]
            maps.append(dd)
    return torch.stack(maps, dim=0)


@torch.no_grad()
def multi_view_point_visibility(
    pts,
    c2ws,
    Ks,
    masks,
    depths=None,
    max_views=None,
    mask_thresh=0.5,
    depth_rel_eps=0.05,
    depth_abs_eps=1e-3,
    require_depth=False,
):
    """
    Soft multi-view visibility with optional Z-buffer occlusion.

    pts: (N,3) NeuS/training frame
    masks: (V,H,W)
    depths: optional (V,H,W) or list — camera-space depth of the *front* surface.
            If provided, a point is visible in view i only if it projects into the
            mask AND is not behind the depth map (z <= z_surf*(1+rel) + abs).

    Returns V ∈ [0,1] shape (N,). no_grad (stopgrad-ready).
    """
    device = pts.device
    dtype = pts.dtype
    c2ws = torch.as_tensor(c2ws, device=device, dtype=dtype)
    Ks = torch.as_tensor(Ks, device=device, dtype=dtype)
    if not torch.is_tensor(masks):
        masks = torch.as_tensor(
            np.stack([np.asarray(m, np.float32) for m in masks]), device=device, dtype=dtype
        )
    else:
        masks = masks.to(device=device, dtype=dtype)
    Vn, H, W = masks.shape
    depth_maps = _stack_depth_maps(depths, H, W, device, dtype)

    n_use = Vn if max_views is None else min(Vn, int(max_views))
    if n_use < Vn:
        idx = torch.linspace(0, Vn - 1, n_use, device=device).long()
        c2ws, Ks, masks = c2ws[idx], Ks[idx], masks[idx]
        if depth_maps is not None:
            depth_maps = depth_maps[idx]
        Vn = n_use

    omega_sum = torch.zeros(pts.shape[0], device=device, dtype=dtype)
    for i in range(Vn):
        u, v, z, in_front = project_points(pts, c2ws[i], Ks[i])
        in_img = (u >= 0) & (u < W - 1) & (v >= 0) & (v < H - 1) & in_front
        ui = u.long().clamp(0, W - 1)
        vi = v.long().clamp(0, H - 1)
        mval = masks[i, vi, ui]
        seen = in_img & (mval > mask_thresh)

        if depth_maps is not None:
            z_surf = depth_maps[i, vi, ui]
            valid_d = z_surf > 1e-4
            # Behind the measured surface → occluded (back face / interior along ray)
            occluded = valid_d & (z > (z_surf * (1.0 + depth_rel_eps) + depth_abs_eps))
            seen = seen & (~occluded)
            if require_depth:
                # no depth at pixel → do not claim visibility
                seen = seen & valid_d
        omega_sum = omega_sum + seen.float()
    return (1.0 - torch.exp(-omega_sum)).clamp(0, 1)


def ray_expected_points(rays_o, rays_d, weights, z):
    depth = (weights * z).sum(dim=-1, keepdim=True) / (weights.sum(dim=-1, keepdim=True) + 1e-6)
    return rays_o + rays_d * depth


def visibility_gate_losses(L_photo, L_unseen, V, eps=1e-4):
    """L = sg(V)*L_photo + (1-sg(V))*L_unseen"""
    if torch.is_tensor(V):
        v = V.detach().float().mean().clamp(0.0, 1.0)
    else:
        v = L_photo.new_tensor(float(V)).clamp(0.0, 1.0)
    return v * L_photo + (1.0 - v) * L_unseen, v


def visibility_weighted_point_loss(pred, target, V):
    """Prior/flow loss weighted by (1-sg(V)) — only where cameras cannot see."""
    w = (1.0 - V.detach().clamp(0, 1))
    err = (pred - target) ** 2
    if err.ndim > 1:
        err = err.reshape(err.shape[0], -1).mean(dim=-1)
    return (w * err).sum() / w.sum().clamp_min(1e-4)


print("✓ multi-view visibility V(x) ready")


### 4.5 — Dual-field ray marcher + learnable poses

Rays accumulate **foreground Hash-SDF density** inside the unit sphere and
**background NeRF** outside (contracted). With `HARD_MASK_ROUTE=True`, SAM-masked
rays update the Hash-SDF; unmasked rays go to the background field only.


In [ ]:
# 4.5 — Cameras (BA) + dual-field volume renderer (mask-routed)
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class LearnablePoses(nn.Module):
    """Axis-angle delta + translation delta on top of init c2w."""

    def __init__(self, c2ws_init: torch.Tensor):
        super().__init__()
        self.register_buffer("c2w0", c2ws_init.float().clone())
        n = c2ws_init.shape[0]
        self.r = nn.Parameter(torch.zeros(n, 3))
        self.t = nn.Parameter(torch.zeros(n, 3))

    def forward(self, idx=None):
        if idx is None:
            idx = torch.arange(self.c2w0.shape[0], device=self.c2w0.device)
        r = self.r[idx]
        angle = r.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        k = r / angle
        Kx = torch.zeros(len(idx), 3, 3, device=r.device, dtype=r.dtype)
        Kx[:, 0, 1], Kx[:, 0, 2] = -k[:, 2], k[:, 1]
        Kx[:, 1, 0], Kx[:, 1, 2] = k[:, 2], -k[:, 0]
        Kx[:, 2, 0], Kx[:, 2, 1] = -k[:, 1], k[:, 0]
        eye = torch.eye(3, device=r.device, dtype=r.dtype).expand(len(idx), 3, 3)
        R_delta = eye + torch.sin(angle)[..., None] * Kx + (1 - torch.cos(angle))[..., None] * (Kx @ Kx)
        R0 = self.c2w0[idx, :3, :3]
        t0 = self.c2w0[idx, :3, 3]
        R = R_delta @ R0
        t = t0 + self.t[idx]
        out = torch.eye(4, device=r.device, dtype=r.dtype).expand(len(idx), 4, 4).clone()
        out[:, :3, :3] = R
        out[:, :3, 3] = t
        return out


def sample_rays(images, masks, depths, Ks, c2ws, n_rays, device, prefer_mask=0.7):
    """Random pixels; prefer_mask fraction drawn from building mask when possible."""
    n_views = len(images)
    H, W = images[0].shape[:2]
    img_t = torch.from_numpy(np.stack(images)).to(device=device, dtype=torch.float32)
    msk_t = torch.from_numpy(np.stack(masks)).to(device=device, dtype=torch.float32)
    dep_stack = [
        np.zeros((H, W), np.float32) if d is None else d.astype(np.float32)
        for d in depths
    ]
    dep_t = torch.from_numpy(np.stack(dep_stack)).to(device=device, dtype=torch.float32)

    n_mask = int(n_rays * prefer_mask)
    n_any = n_rays - n_mask
    vi_list, ys_list, xs_list = [], [], []
    flat = msk_t.reshape(n_views, -1)

    if n_mask > 0:
        vi_m = torch.randint(0, n_views, (n_mask,), device=device)
        ys_m = torch.randint(0, H, (n_mask,), device=device)
        xs_m = torch.randint(0, W, (n_mask,), device=device)
        ok = msk_t[vi_m, ys_m, xs_m] > 0.5
        if (~ok).any():
            v0 = int(torch.randint(0, n_views, (1,)).item())
            probs = flat[v0].clamp_min(0)
            if float(probs.sum()) > 0:
                pick = torch.multinomial(probs, int((~ok).sum().item()), replacement=True)
                ys_m[~ok] = pick // W
                xs_m[~ok] = pick % W
                vi_m[~ok] = v0
        vi_list.append(vi_m); ys_list.append(ys_m); xs_list.append(xs_m)

    if n_any > 0:
        vi_list.append(torch.randint(0, n_views, (n_any,), device=device))
        ys_list.append(torch.randint(0, H, (n_any,), device=device))
        xs_list.append(torch.randint(0, W, (n_any,), device=device))

    vi = torch.cat(vi_list)[:n_rays]
    ys = torch.cat(ys_list)[:n_rays]
    xs = torch.cat(xs_list)[:n_rays]

    rgb = img_t[vi, ys, xs]
    msk = msk_t[vi, ys, xs]
    dep = dep_t[vi, ys, xs]
    K = torch.as_tensor(Ks, device=device, dtype=torch.float32)[vi]
    c2w = torch.as_tensor(c2ws, device=device, dtype=torch.float32)[vi]
    fx, fy = K[:, 0, 0], K[:, 1, 1]
    cx, cy = K[:, 0, 2], K[:, 1, 2]
    dirs_c = torch.stack([
        (xs.float() - cx) / fx,
        (ys.float() - cy) / fy,
        torch.ones_like(fx),
    ], dim=-1)
    dirs_c = dirs_c / (dirs_c.norm(dim=-1, keepdim=True) + 1e-8)
    rays_d = (c2w[:, :3, :3] @ dirs_c.unsqueeze(-1)).squeeze(-1)
    rays_o = c2w[:, :3, 3]
    return rays_o, rays_d, rgb, msk, dep, vi


def near_far_from_sphere(rays_o, rays_d, radius=1.0):
    a = (rays_d * rays_d).sum(-1)
    b = 2 * (rays_o * rays_d).sum(-1)
    mid = -b / (2 * a + 1e-8)
    return (mid - radius).clamp_min(0.01), mid + radius


def render_dual_field(
    rays_o, rays_d,
    sdf_net, bg_net, color_net,
    gamma=None, n_samples=64, n_bg=32, mask_rays=None, hard_mask_route=True,
):
    """
    Dual-field render. hard_mask_route: FG density only on SAM-masked rays.
    """
    device = rays_o.device
    n_rays = rays_o.shape[0]
    near, far = near_far_from_sphere(rays_o, rays_d)

    t_vals = torch.linspace(0, 1, n_samples, device=device)
    z = near[:, None] + (far - near)[:, None] * t_vals[None, :]
    mids = 0.5 * (z[:, 1:] + z[:, :-1])
    upper = torch.cat([mids, z[:, -1:]], -1)
    lower = torch.cat([z[:, :1], mids], -1)
    z = lower + (upper - lower) * torch.rand_like(z)

    pts = rays_o[:, None, :] + rays_d[:, None, :] * z[..., None]
    pts_flat = pts.reshape(-1, 3)
    dirs_flat = rays_d[:, None, :].expand_as(pts).reshape(-1, 3)

    pts_flat.requires_grad_(True)
    dens, sdf, feat = sdf_net.density(pts_flat, gamma=gamma)
    grad = torch.autograd.grad(sdf.sum(), pts_flat, create_graph=True, retain_graph=True)[0]
    eik = ((grad.norm(dim=-1) - 1.0) ** 2).mean()
    normal = F.normalize(grad, dim=-1).detach()

    rgb_fg = color_net(pts_flat, normal, dirs_flat, feat).reshape(n_rays, n_samples, 3)
    dens = dens.reshape(n_rays, n_samples)
    sdf = sdf.reshape(n_rays, n_samples)

    deltas = z[:, 1:] - z[:, :-1]
    deltas = torch.cat([deltas, torch.full_like(deltas[:, :1], 1e-2)], dim=-1)
    dens = dens * (pts.norm(dim=-1) < 1.0).float()

    if hard_mask_route and mask_rays is not None:
        dens = dens * mask_rays.float().clamp(0, 1)[:, None]

    w_fg, T_fg = rendering_weights_along_ray(dens, deltas)
    rgb_fg_acc = (w_fg[..., None] * rgb_fg).sum(dim=1)
    depth_fg = (w_fg * z).sum(dim=1)
    O_fg = w_fg.sum(dim=-1).clamp(0, 1)

    z_bg = far[:, None] + torch.linspace(0, 1, n_bg, device=device)[None, :] * 2.0
    pts_bg = rays_o[:, None, :] + rays_d[:, None, :] * z_bg[..., None]
    dens_bg, rgb_bg = bg_net(pts_bg.reshape(-1, 3))
    dens_bg = dens_bg.reshape(n_rays, n_bg)
    rgb_bg = rgb_bg.reshape(n_rays, n_bg, 3)
    dlt_bg = z_bg[:, 1:] - z_bg[:, :-1]
    dlt_bg = torch.cat([dlt_bg, torch.full_like(dlt_bg[:, :1], 1e-1)], dim=-1)
    w_bg, _ = rendering_weights_along_ray(dens_bg, dlt_bg)
    rgb_bg_acc = (w_bg[..., None] * rgb_bg).sum(dim=1)
    rgb = rgb_fg_acc + (1.0 - O_fg)[:, None] * rgb_bg_acc

    return {
        "rgb": rgb, "depth": depth_fg, "O_fg": O_fg, "weights_fg": w_fg,
        "eikonal": eik, "sdf": sdf, "z": z, "pts": pts, "dens": dens,
    }


print("✓ renderer + learnable poses ready (mask-routed)")


## 5 — Smoke tests (no training)

Verifies hash annealing, density conversion, proxy extraction, flow loss
backward into hash params, and a single render step.


In [ ]:
# 5.0 — Module smoke tests
import traceback
import numpy as np
import torch.nn.functional as F

device = torch.device(DEVICE)
enc = HashGridEncoding(
    n_levels=HASH_N_LEVELS, n_features_per_level=HASH_N_FEATURES,
    log2_hashmap_size=min(HASH_LOG2_SIZE, 16),
    base_resolution=HASH_BASE_RES, max_resolution=512,
).to(device)
sdf_net = HashSDF(enc, hidden=SDF_HIDDEN, n_layers=SDF_N_LAYERS).to(device)
bg_net = BackgroundNeRF(hidden=BG_HIDDEN).to(device)
color_net = ColorNet(feat_dim=SDF_HIDDEN, hidden=SDF_HIDDEN).to(device)

g = anneal_gamma(1000, HASH_N_LEVELS, ANNEAL_ITERS)
W = hash_anneal_weights(HASH_N_LEVELS, g)
print(f"[anneal] γ(1000)={g:.2f} | active levels≈{(W > 0.01).sum().item()} | W[:6]={W[:6].cpu().numpy().round(3)}")

pts = torch.rand(128, 3, device=device) * 2 - 1
dens, sdf, feat = sdf_net.density(pts, gamma=g)
print(f"[density] sdf∈[{sdf.min():.3f},{sdf.max():.3f}] dens∈[{dens.min():.3f},{dens.max():.3f}] β={sdf_net.beta.detach().item():.2f}")

V = multi_view_point_visibility(
    pts.detach(), scene["c2ws"], scene["Ks"],
    torch.from_numpy(np.stack(scene["masks"]).astype(np.float32)),
    depths=scene.get("depths"),
    max_views=min(4, scene["n_views"]),
)
print(f"[vis] V∈[{float(V.min()):.3f},{float(V.max()):.3f}] mean={float(V.mean()):.3f}")

_occ = OCC_GRID_N11.to(device) if (OCC_GRID_N11 is not None) else None
try:
    prior = build_flow_prior(
        "occupancy" if _occ is not None else "stub",
        occ_grid_n11=_occ,
    ).to(device)
    L_flow, info = flow_distillation_loss(sdf_net, prior, res=16, gamma=g, vis_weight=None)
    L_flow.backward()
    grad_ok = any(p.grad is not None and p.grad.abs().sum() > 0 for p in enc.parameters())
    print(f"[flow] L={float(L_flow):.5f} t={info['t']:.3f} grad_reaches_hash={grad_ok} mode={FLOW_PRIOR_MODE}")
except Exception as e:
    print("[flow] skipped:", e)
    traceback.print_exc()

sdf_net.zero_grad(set_to_none=True)
poses = LearnablePoses(torch.from_numpy(scene["c2ws"]).float()).to(device)
c2ws = poses()
rays_o, rays_d, rgb_gt, msk, dep, vi = sample_rays(
    scene["images"], scene["masks"], scene["depths"], scene["Ks"], c2ws, n_rays=64, device=device
)
out = render_dual_field(
    rays_o, rays_d, sdf_net, bg_net, color_net, gamma=g, n_samples=32, n_bg=16,
    mask_rays=msk, hard_mask_route=True,
)
L_photo = F.mse_loss(out["rgb"], rgb_gt)
L_mask = F.binary_cross_entropy(out["O_fg"].clamp(1e-4, 1 - 1e-4), msk.clamp(0, 1))
L_tot, v_used = visibility_gate_losses(L_photo, torch.tensor(0.01, device=device), V.mean())
(L_tot + 0.1 * out["eikonal"] + 0.5 * L_mask).backward()
print(f"[render] photo={float(L_photo):.4f} mask={float(L_mask):.4f} Vgate={float(v_used):.3f} O_fg={float(out['O_fg'].mean()):.3f}")
print("✓ all smoke tests passed")


## 6 — Train GF-HashBA

Each iteration:

1. Sample rays; render dual-field with current $\gamma(k)$
2. $\mathcal{L}_{\mathrm{photo}}$, $\mathcal{L}_{\mathrm{mask}}$, $\mathcal{L}_{\mathrm{eik}}$, optional depth
3. On adaptive Stage-2 cadence (~12 early → ~100 late): extract $z_0$, apply flow distillation
4. Gate with $\mathrm{sg}(V)$: visible → photo, invisible → flow (unless `FLOW_PRIOR_ON_VISIBLE`)
5. Step SDF / color / bg / pose optimizers


### 6.0b — Prefit Hash-SDF to Stage-1 occupancy

Same idea as BA-NeuS §13.2: before photometric training, fit the SDF to
`pseudo_sdf = (0.5 - p) * 0.2` from Stage-1. Gives a building-shaped zero level
set instead of a random hash field. §6.0 calls this automatically when
`USE_STAGE1_PREFIT=True`.


In [ ]:
# 6.0b — Stage-1 SDF prefit helper (called from §6.0)
import time
import shutil
import numpy as np
import torch.nn.functional as F


def prefit_hash_sdf(
    sdf_net,
    n_iters=SDF_PREFIT_ITERS,
    batch=SDF_PREFIT_BATCH,
    lr=SDF_PREFIT_LR,
):
    """Fit Hash-SDF to Stage-1 pseudo-SDF inside the unit ball."""
    if not HAS_STAGE1:
        print("skip prefit — no Stage-1")
        return None

    # Diagnose teacher BEFORE fitting — mse→0 with empty/constant target is a false win
    with torch.no_grad():
        _probe = torch.randn(8192, 3, device=device)
        _probe = _probe / (_probe.norm(dim=-1, keepdim=True) + 1e-6) * torch.rand(8192, 1, device=device) ** (1 / 3)
        _p = sample_prob(_probe)
        _tgt = pseudo_sdf_from_prob(_p)
        print(
            f"Stage-1 teacher probe: occ mean={float(_p.mean()):.3f}  "
            f"frac>0.5={float((_p > 0.5).float().mean()):.3f}  "
            f"pseudo-SDF∈[{float(_tgt.min()):.3f},{float(_tgt.max()):.3f}] mean={float(_tgt.mean()):.3f}"
        )
        if float((_p > 0.5).float().mean()) < 0.01:
            print("⚠ Almost no occupied voxels in NeuS ball — Stage-1↔frame mapping is wrong.")
            print("  Prefit will learn ~empty SDF. Fix sam_baneus §12.1b/§12.2 before trusting this run.")

    opt = torch.optim.Adam(sdf_net.parameters(), lr=lr)
    sdf_net.train()
    t0 = time.time()
    last = 0.0
    for it in range(n_iters):
        pts = torch.randn(batch, 3, device=device)
        pts = pts / (pts.norm(dim=-1, keepdim=True) + 1e-6) * torch.rand(batch, 1, device=device) ** (1 / 3)
        with torch.no_grad():
            tgt = pseudo_sdf_from_prob(sample_prob(pts)).unsqueeze(-1)
        g = anneal_gamma(it, HASH_N_LEVELS, max(1, n_iters // 2), ANNEAL_START_LVL)
        pred, _ = sdf_net(pts, gamma=g)
        loss = F.mse_loss(pred, tgt)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        last = float(loss)
        if it % 500 == 0 or it == n_iters - 1:
            print(f"  prefit {it:5d}/{n_iters}  mse={last:.5f}  γ={g:.2f}  [{time.time()-t0:.0f}s]")

    path = OUTPUT_DIR / "gfhashba_sdf_prefit.pth"
    torch.save({"sdf": sdf_net.state_dict(), "n_iters": n_iters, "mse": last}, path)
    print("saved", path)

    # ---- extract + upload prefit mesh (survives disconnect) ----
    try:
        from skimage.measure import marching_cubes
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-image"])
        from skimage.measure import marching_cubes
    import matplotlib.pyplot as plt
    import trimesh

    sdf_net.eval()
    RES = 128
    lin = torch.linspace(-1, 1, RES, device=device)
    zz, yy, xx = torch.meshgrid(lin, lin, lin, indexing="ij")
    pts = torch.stack([xx, yy, zz], dim=-1).reshape(-1, 3)
    vols = []
    with torch.no_grad():
        for i in range(0, pts.shape[0], 65536):
            s, _ = sdf_net(pts[i:i + 65536], gamma=float(HASH_N_LEVELS))
            vols.append(s.squeeze(-1).float().cpu())
    vol = torch.cat(vols).reshape(RES, RES, RES).numpy()
    print(f"prefit SDF range [{vol.min():.3f},{vol.max():.3f}]")
    try:
        verts, faces, normals, _ = marching_cubes(vol, level=0.0)
        verts = verts / (RES - 1) * 2.0 - 1.0
        mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
        out_glb = OUTPUT_DIR / "gfhashba_prefit_mesh.glb"
        out_ply = OUTPUT_DIR / "gfhashba_prefit_mesh.ply"
        mesh.export(out_glb); mesh.export(out_ply)
        print(f"prefit mesh: {len(mesh.vertices)} verts → {out_glb}")

        fig = plt.figure(figsize=(10, 3.5))
        for j, (az, el) in enumerate([(-60, 20), (40, 25), (90, 75)]):
            ax = fig.add_subplot(1, 3, j + 1, projection="3d")
            ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2], triangles=faces,
                            cmap="viridis", linewidth=0, antialiased=False)
            ax.view_init(elev=el, azim=az); ax.set_box_aspect((1, 1, 1))
            ax.set_title(f"prefit ({az},{el})")
        plt.tight_layout()
        out_png = OUTPUT_DIR / "gfhashba_prefit_preview.png"
        fig.savefig(out_png, dpi=120, bbox_inches="tight")
        plt.show()
    except Exception as e:
        print("prefit MC failed:", e)
        out_glb = out_ply = out_png = None

    if Path("/content/drive/MyDrive").exists():
        DRIVE_OUT.mkdir(parents=True, exist_ok=True)
        for p in (path, out_glb, out_ply, out_png):
            if p is not None and Path(p).exists():
                shutil.copy2(p, DRIVE_OUT / Path(p).name)
        print("→ Drive", DRIVE_OUT)

    sdf_net.train()
    return path


print("prefit helper ready (exports mesh → Drive)")


### 6.0a — Load real MV-SAM3D Stage-2 velocity prior

Install path **mirrors** `mvsam3d_stage2LoRA` / `sam_baneus_ultrra` / `mvsam3d_example_current`:
`PIP_CONSTRAINT` (numpy 1.26.4 + scipy 1.13.1) → same pip dep list → Drive
`colab_compiled_pkgs.tar.gz` → companion wheels (`pygltflib`, `pccm`) → hydra patch →
`Inference` → teacher SLAT via `inject_generator_multi_view` → `Stage2FlowPrior`.

Requires **torch 2.5.1+cu121** (§1.0b). Missing `pygltflib`/`pccm` after tarball is normal
(not an ABI error) — this cell installs them.


In [ ]:
# 6.0a — Build Stage-2 Flow prior (same path as mvsam3d_stage2LoRA)
# Uses: Inference + slat_generator + inject_generator_multi_view + reverse_fn (FlowMatching)
import os, sys, importlib.util, subprocess
import numpy as np
import torch
from pathlib import Path
from PIL import Image

STAGE2_PRIOR = None


def _ensure_mvsam3d_stack():
    """Clone / link / resolve paths exactly like mvsam3d_stage2LoRA §§1–2 + load cell."""
    root = Path("/content")
    sam3d = root / "sam-3d-objects"
    mvsam = root / "MV-SAM3D"

    repos = {
        "sam-3d-objects": "https://github.com/facebookresearch/sam-3d-objects.git",
        "MV-SAM3D": "https://github.com/devinli123/MV-SAM3D.git",
    }
    cwd = os.getcwd()
    try:
        os.chdir(str(root))
        for name, url in repos.items():
            if (root / name).is_dir():
                print(f"OK   {name}/")
            else:
                print(f"cloning {name}…")
                subprocess.check_call(["git", "clone", "--depth", "1", url, name])
    finally:
        os.chdir(cwd)

    inf = mvsam / "notebook" / "inference.py"
    if not inf.exists():
        (mvsam / "notebook").mkdir(parents=True, exist_ok=True)
        for fn in ("inference.py", "load_images_and_masks.py", "mesh_alignment.py"):
            src, dst = sam3d / "notebook" / fn, mvsam / "notebook" / fn
            if src.exists() and not dst.exists():
                os.symlink(src, dst)
                print("linked", dst)

    candidates = [
        mvsam,
        Path("/content/drive/MyDrive/SatelliteFuse/code/MV-SAM3D"),
        sam3d,
    ]
    mvsam_root, inf_py = None, None
    for r in candidates:
        cand = r / "notebook" / "inference.py"
        if cand.exists():
            mvsam_root, inf_py = r, cand
            break
    if inf_py is None:
        raise FileNotFoundError(
            "notebook/inference.py not found after clone/link.\n"
            f"Checked: {[str(c / 'notebook' / 'inference.py') for c in candidates]}"
        )

    try:
        from huggingface_hub import login, snapshot_download
        try:
            from google.colab import userdata
            hf_token = userdata.get("HF_TOKEN")
        except Exception:
            hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
        if hf_token:
            login(token=hf_token)
        ckpt_hf = sam3d / "checkpoints" / "hf"
        need_dl = not (
            (ckpt_hf / "pipeline.yaml").exists()
            or (ckpt_hf / "checkpoints" / "pipeline.yaml").exists()
        )
        if need_dl:
            if not hf_token:
                raise RuntimeError("Need Colab secret HF_TOKEN for facebook/sam-3d-objects")
            print("Downloading facebook/sam-3d-objects checkpoints…")
            snapshot_download(repo_id="facebook/sam-3d-objects", token=hf_token, local_dir=str(ckpt_hf))
        dst = mvsam / "checkpoints" / "hf"
        src_nested = ckpt_hf / "checkpoints"
        src = src_nested if (src_nested / "pipeline.yaml").exists() else ckpt_hf
        if not dst.exists() and src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            os.symlink(src, dst)
            print("symlinked", dst, "→", src)
    except Exception as e:
        print(f"⚠ checkpoint ensure: {type(e).__name__}: {e}")

    pipe_yaml = mvsam / "checkpoints" / "hf" / "pipeline.yaml"
    if not pipe_yaml.exists():
        pipe_yaml = mvsam_root / "checkpoints" / "hf" / "pipeline.yaml"
    if not pipe_yaml.exists():
        for alt in (
            sam3d / "checkpoints" / "hf" / "pipeline.yaml",
            sam3d / "checkpoints" / "hf" / "checkpoints" / "pipeline.yaml",
        ):
            if alt.exists():
                pipe_yaml = alt
                break
    if not pipe_yaml.exists():
        raise FileNotFoundError(f"pipeline.yaml missing under {mvsam}/checkpoints/hf — set HF_TOKEN")

    nb_dir = str(inf_py.parent)
    if nb_dir not in sys.path:
        sys.path.insert(0, nb_dir)
    if str(mvsam_root) not in sys.path:
        sys.path.insert(0, str(mvsam_root))

    print("MVSAM3D_ROOT =", mvsam_root)
    print("inference.py =", inf_py)
    print("PIPELINE_YAML =", pipe_yaml)
    return mvsam_root, inf_py, pipe_yaml


def _ensure_sam3d_python_deps():
    """Exact env path from sam_baneus_ultrra + mvsam3d_stage2LoRA + mvsam3d_example_current.

    Critical order (do not reorder):
      1) PIP_CONSTRAINT + numpy/scipy pin
      2) pip core deps (LoRA list) + warp-lang companions
      3) CUMM_DISABLE_JIT=1 / SPCONV_DISABLE_JIT=1  BEFORE any cumm/spconv import
      4) restore colab_compiled_pkgs.tar.gz
      5) re-pin numpy/scipy
      6) import pytorch3d, flash_attn, kaolin, gsplat, spconv.pytorch
    """
    import site
    import importlib

    def _pip(*args):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

    import torch
    TARGET = "2.5.1"
    print(f"torch={torch.__version__} (Stage-2 needs {TARGET}+cu121) py={sys.version.split()[0]}")
    if sys.version_info >= (3, 13):
        raise RuntimeError(
            f"Python {sys.version.split()[0]} has no torch 2.5.1 wheels. "
            "Pin Colab Runtime version to 2026.07/2026.04 (Python 3.12)."
        )
    if not torch.__version__.startswith(TARGET):
        # Install one-by-one so torchvision cannot fall back to bare PyPI (0.1.6/0.2.0 only).
        cu = "https://download.pytorch.org/whl/cu121"
        for pkg in ("torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1"):
            _pip(pkg, "--index-url", cu)
        try:
            _pip("xformers==0.0.28.post3", "--index-url", cu)
        except Exception:
            pass
        raise RuntimeError(
            f"Torch pinned to {TARGET}. Runtime → Restart session, then re-run from §1.1 / §6.0a."
        )

    pin = Path("/content/_numpy_pin.txt")
    pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
    os.environ["PIP_CONSTRAINT"] = str(pin)
    print("PIP_CONSTRAINT ->", os.environ["PIP_CONSTRAINT"])
    _pip("numpy==1.26.4", "scipy==1.13.1")
    import numpy, scipy
    print(f"✓ numpy {numpy.__version__}  scipy {scipy.__version__}")

    try:
        import sam3d_objects  # noqa: F401
    except Exception:
        _pip("--no-deps", "-e", "/content/sam-3d-objects")

    print("pip: LoRA/sam_baneus core deps…")
    _pip(
        "hydra-core==1.3.2",
        "omegaconf",
        "loguru",
        "opencv-python-headless",
        "trimesh",
        "roma",
        "seaborn",
        "safetensors",
        "tqdm",
        "einops",
        "astor",
        "lightning",
        "xatlas",
        "pyvista",
        "pymeshfix",
        "igraph",
        "numpy<2",
        "huggingface_hub[cli]",
        "gradio",
        "peft==0.13.2",
        "accelerate",
        "matplotlib",
        "pillow",
        "pygltflib",
        "pccm",
        "addict",
        "yapf",
        "warp-lang",
        "ipycanvas",
        "ipyevents",
        "plyfile",
        "easydict",
        "jaxtyping",
        "timm",
        "imageio",
        "scikit-image",
    )
    try:
        _pip("open3d")
    except Exception as e:
        print("⚠ open3d:", e)

    try:
        import utils3d  # noqa: F401
    except Exception:
        _pip(
            "--no-deps",
            "utils3d @ git+https://github.com/EasternJournalist/utils3d.git@3913c65d81e05e47b9f367250cf8c0f7462a0900",
        )
    try:
        import MoGe  # noqa: F401
    except Exception:
        try:
            import moge  # noqa: F401
        except Exception:
            _pip(
                "--no-deps",
                "MoGe @ git+https://github.com/microsoft/MoGe.git@a8c37341bc0325ca99b9d57981cc3bb2bd3e255b",
            )

    # ── sam_baneus: MUST set before ANY cumm/spconv import (reads env once) ──
    os.environ["CUMM_DISABLE_JIT"] = "1"
    os.environ["SPCONV_DISABLE_JIT"] = "1"
    print("CUMM_DISABLE_JIT=1 SPCONV_DISABLE_JIT=1 (sam_baneus — skip bogus ninja rebuild)")

    def _purge_compiled():
        for name in list(sys.modules):
            root = name.split(".")[0]
            if root in ("pytorch3d", "flash_attn", "kaolin", "gsplat", "spconv", "cumm", "warp"):
                del sys.modules[name]

    def _compiled_ok():
        """Same spirit as LoRA _compiled_ok, but probe spconv.pytorch (sam_baneus)."""
        try:
            import pytorch3d  # noqa: F401
            import flash_attn  # noqa: F401
            import kaolin  # noqa: F401
            import gsplat  # noqa: F401
            import spconv.pytorch  # noqa: F401
            return None
        except Exception as e:
            return e

    # example_current / LoRA: restore tarball FIRST — never import spconv before that
    # (importing a broken/partial cumm triggers ninja rebuild → tensorview/*.h errors).
    sp = site.getsitepackages()[0]
    backup = next(
        (
            b
            for b in (
                Path("/content/drive/MyDrive/colab_compiled_pkgs.tar.gz"),
                Path("/content/drive/MyDrive/SatelliteFuse/colab_compiled_pkgs.tar.gz"),
            )
            if b.exists()
        ),
        None,
    )
    if backup is None:
        raise RuntimeError(
            "No colab_compiled_pkgs.tar.gz. Run mvsam3d_stage2LoRA §§0–4c or "
            "mvsam3d_example_current restore once, then retry."
        )

    print("Restoring compiled pkgs from", backup)
    subprocess.check_call(["tar", "xzf", str(backup), "-C", sp])
    _pip("numpy==1.26.4", "scipy==1.13.1")
    # re-assert JIT flags after any import side effects
    os.environ["CUMM_DISABLE_JIT"] = "1"
    os.environ["SPCONV_DISABLE_JIT"] = "1"
    _purge_compiled()

    # Pure-python companions ONLY — never pip cumm/spconv/pytorch3d (tarball owns those).
    # Bug: auto `pip cumm` on ModuleNotFoundError destroyed prebuilt cumm.core_cc.
    _SAFE_PIP = {
        "warp": "warp-lang",
        "ipycanvas": "ipycanvas",
        "ipyevents": "ipyevents",
        "pygltflib": "pygltflib",
        "pccm": "pccm",
        "addict": "addict",
        "plyfile": "plyfile",
        "easydict": "easydict",
        "jaxtyping": "jaxtyping",
    }
    _TAR_ONLY = {"cumm", "spconv", "pytorch3d", "flash_attn", "kaolin", "gsplat", "core_cc"}

    def _restore_tarball():
        print("Restoring compiled pkgs from", backup)
        subprocess.check_call(["tar", "xzf", str(backup), "-C", sp])
        _pip("numpy==1.26.4", "scipy==1.13.1")
        os.environ["CUMM_DISABLE_JIT"] = "1"
        os.environ["SPCONV_DISABLE_JIT"] = "1"
        _purge_compiled()

    err = _compiled_ok()
    if err is not None and isinstance(err, ModuleNotFoundError):
        miss = getattr(err, "name", None) or ""
        root = miss.split(".")[0] if miss else ""
        if not root and "No module named" in str(err):
            import re
            m = re.search(r"No module named ['\"]([^'\"]+)['\"]", str(err))
            root = m.group(1).split(".")[0] if m else ""
        if root in _TAR_ONLY or root.startswith("cumm"):
            print(f"compiled missing '{miss}' — re-extract tarball (do NOT pip {root})")
            _restore_tarball()
            err = _compiled_ok()
        elif root in _SAFE_PIP:
            print(f"missing {root} → pip {_SAFE_PIP[root]}")
            _pip(_SAFE_PIP[root])
            _purge_compiled()
            err = _compiled_ok()
        else:
            print(f"⚠ unknown missing module '{root}': {err}")

    if err is not None and ("cumm" in str(err).lower() or "spconv" in str(err).lower()):
        print("cumm/spconv still broken — clean tarball re-extract (no pip cumm)")
        _restore_tarball()
        err = _compiled_ok()

    if err is not None:
        raise RuntimeError(
            "compiled stack still broken after tarball + CUMM_DISABLE_JIT.\n"
            f"{type(err).__name__}: {err}\n"
            "If you previously saw 'missing cumm → pip cumm', the kernel is poisoned:\n"
            "Runtime → Restart session, then §1.0b → … → §6.0a (fixed notebook)."
        )
    print("✓ compiled stack OK (pytorch3d/flash_attn/kaolin/gsplat/spconv.pytorch)")


def ensure_peft_for_lora():
    try:
        import peft
        print(f"✓ peft {getattr(peft, '__version__', '?')}")
        return peft
    except Exception as e:
        print(f"peft import failed ({e}); pip peft==0.13.2 …")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "peft==0.13.2"]
        )
        for name in list(sys.modules):
            if name.split(".")[0] in ("peft", "transformers"):
                del sys.modules[name]
        import peft
        return peft


def _load_inference(inf_py, pipe_yaml):
    # Colab has no conda — inference.py does: CUDA_HOME = CONDA_PREFIX
    cuda_home = "/usr/local/cuda"
    if not Path(cuda_home).exists():
        for cand in ("/usr/local/cuda-12.1", "/usr/local/cuda-12", "/usr/local/cuda-11.8"):
            if Path(cand).exists():
                cuda_home = cand
                break
    os.environ["CUDA_HOME"] = cuda_home
    os.environ["CONDA_PREFIX"] = cuda_home
    os.environ["LIDRA_SKIP_INIT"] = "true"
    # sam_baneus: must precede any accidental cumm import inside deps
    os.environ["CUMM_DISABLE_JIT"] = "1"
    os.environ["SPCONV_DISABLE_JIT"] = "1"

    _ensure_sam3d_python_deps()

    hydra_patch = Path("/content/sam-3d-objects/patching/hydra")
    if hydra_patch.exists():
        try:
            subprocess.check_call([sys.executable, str(hydra_patch)])
            print("✓ Hydra patched")
        except Exception as e:
            print(f"⚠ hydra patch: {e}")

    # Final gate + Inference import with auto-pip for remaining pure ModuleNotFoundErrors
    _ALIAS = {
        "plyfile": "plyfile",
        "easydict": "easydict",
        "jaxtyping": "jaxtyping",
        "timm": "timm",
        "warp": "warp-lang",
        "ipycanvas": "ipycanvas",
        "imageio": "imageio",
        "skimage": "scikit-image",
        "cv2": "opencv-python-headless",
    }

    def _pip_miss(name):
        pkg = _ALIAS.get(name, name)
        print(f"  Inference missing '{name}' → pip {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

    def _load_inference_cls():
        try:
            from inference import Inference as _Inf
            return _Inf
        except ModuleNotFoundError:
            spec = importlib.util.spec_from_file_location("mvsam3d_notebook_inference", inf_py)
            mod = importlib.util.module_from_spec(spec)
            sys.modules["mvsam3d_notebook_inference"] = mod
            spec.loader.exec_module(mod)
            return mod.Inference

    Inference = None
    last_err = None
    for _round in range(15):
        try:
            import hydra, utils3d, kaolin  # noqa: F401
            Inference = _load_inference_cls()
            last_err = None
            break
        except ModuleNotFoundError as e:
            last_err = e
            miss = getattr(e, "name", None) or ""
            miss = miss.split(".")[0] if miss else ""
            if not miss:
                import re
                m = re.search(r"No module named ['\"]([^'\"]+)['\"]", str(e))
                miss = m.group(1).split(".")[0] if m else ""
            if not miss:
                raise
            _pip_miss(miss)
            # drop half-imported sam3d / inference modules so retry is clean
            for name in list(sys.modules):
                if name.split(".")[0] in ("inference", "sam3d_objects", "mvsam3d_notebook_inference", miss):
                    del sys.modules[name]
        except Exception as e:
            raise RuntimeError(
                f"Pre-Inference import failed: {type(e).__name__}: {e}\n"
                "Deps should match mvsam3d_stage2LoRA / sam_baneus_ultrra."
            ) from e
    if Inference is None:
        raise RuntimeError(f"Could not import Inference after retries: {last_err}")

    print("Loading Inference pipeline (Stage-2 / FlowMatching)…")
    inference = Inference(str(pipe_yaml), compile=False)
    return inference



def _as_alpha(mask) -> np.ndarray:
    m = np.asarray(mask)
    if m.ndim == 3:
        m = m[..., 3] if m.shape[-1] == 4 else m[..., 0]
    if m.dtype == bool:
        m = m.astype(np.uint8) * 255
    elif m.dtype != np.uint8:
        m = (m * 255).astype(np.uint8) if float(m.max()) <= 1.0 else m.astype(np.uint8)
    return m[..., None]


def _preprocess_views(pipeline, view_images, view_masks):
    """Same RGBA + ss/slat preprocess as mvsam3d_stage2LoRA."""
    ss_dicts, slat_dicts = [], []
    for image, mask in zip(view_images, view_masks):
        image = np.asarray(image)[..., :3]
        if mask is not None:
            rgba = np.concatenate([image.astype(np.uint8), _as_alpha(mask)], axis=-1)
        else:
            rgba = np.concatenate(
                [image.astype(np.uint8), np.full((*image.shape[:2], 1), 255, np.uint8)], axis=-1
            )
        rgba_pil = Image.fromarray(rgba, mode="RGBA")
        if hasattr(pipeline, "compute_pointmap"):
            pmd = pipeline.compute_pointmap(rgba_pil, pointmap=None)
            pm = pmd["pointmap"]
            ss_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.ss_preprocessor, pointmap=pm))
            slat_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.slat_preprocessor))
        else:
            ss_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.ss_preprocessor))
            slat_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.slat_preprocessor))
    return ss_dicts, slat_dicts


def build_stage2_flow_prior(micro_n=None, teacher_steps=None, lora_dir=None, seed=0):
    """Teacher-sample SLAT via inject_generator_multi_view; wrap reverse_fn as Stage2FlowPrior."""
    global STAGE2_PRIOR
    micro_n = int(micro_n if micro_n is not None else STAGE2_MICRO_N)
    teacher_steps = int(teacher_steps if teacher_steps is not None else STAGE2_TEACHER_STEPS)
    lora_dir = lora_dir if lora_dir is not None else STAGE2_LORA_DIR

    _mvsam_root, inf_py, pipe_yaml = _ensure_mvsam3d_stack()
    inference = _load_inference(inf_py, pipe_yaml)
    pipeline = inference._pipeline
    device_s2 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    slat_gen = pipeline.models["slat_generator"]
    cfg_mod = slat_gen.reverse_fn
    print("slat_generator:", type(slat_gen).__name__, "| reverse_fn:", type(cfg_mod).__name__)

    if lora_dir is not None:
        ld = Path(str(lora_dir))
        if ld.exists():
            peft = ensure_peft_for_lora()
            from peft import PeftModel
            print("Loading Stage-2 LoRA from", ld)
            slat_gen.reverse_fn.backbone = PeftModel.from_pretrained(
                slat_gen.reverse_fn.backbone, str(ld)
            )
            for p in slat_gen.parameters():
                p.requires_grad_(False)
        else:
            print(f"⚠ STAGE2_LORA_DIR missing ({ld}) — using base Stage-2 weights")

    img_dir = AERIAL_DIR / IMG_SUBDIR
    mask_dir = AERIAL_DIR / MASK_SUBDIR

    def _nat(p):
        s = Path(p).stem
        return (0, int(s), s) if s.isdigit() else (1, 0, s)

    paths = sorted(
        [p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}],
        key=_nat,
    )[:micro_n]
    assert paths, f"No images in {img_dir}"
    view_images, view_masks = [], []
    for ip in paths:
        view_images.append(Image.open(ip).convert("RGB"))
        mp = mask_dir / f"{ip.stem}.png"
        if not mp.exists():
            cands = list(mask_dir.glob(ip.stem + "*"))
            mp = cands[0] if cands else None
        view_masks.append(Image.open(mp) if mp is not None else None)
    print("Stage-2 prior views:", [p.stem for p in paths])

    torch.manual_seed(seed)
    ss_dicts, slat_dicts = _preprocess_views(pipeline, view_images, view_masks)

    from sam3d_objects.pipeline.multi_view_utils import inject_generator_multi_view

    if STAGE1_ALIGN is None:
        raise RuntimeError("STAGE1_ALIGN missing — run §3.5 before §6.0a (need baneus_scene_meta Sim3)")

    use_export = bool(globals().get("USE_EXPORTED_STAGE1_COORDS", True)) and (STAGE1_COORDS is not None)
    if use_export:
        coords = STAGE1_COORDS.to(device=device_s2, dtype=torch.int32)
        print(f"Stage-1 coords from export (same as prefit): {tuple(coords.shape)} dtype={coords.dtype}")
    else:
        print("Stage-1 multi-view → coords (fresh SS — may disagree with prefit)…")
        with torch.no_grad():
            ss_ret = pipeline.sample_sparse_structure_multi_view(
                ss_dicts, inference_steps=25, mode="multidiffusion",
            )
        coords = ss_ret["coords"]
        print("coords", tuple(coords.shape))
    # spconv SparseTensor asserts indices dtype == int32
    if torch.is_tensor(coords):
        coords = coords.to(dtype=torch.int32)
    else:
        coords = torch.as_tensor(coords, dtype=torch.int32, device=device_s2)

    print("Teacher Stage-2 (inject_generator_multi_view) → x1…")
    slat_gen.no_shortcut = True
    cfg_mod.strength = float(getattr(pipeline, "slat_cfg_strength", 0.0) or 0.0)
    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=pipeline.dtype, enabled=(device_s2.type == "cuda")):
            cond_args, cond_kwargs = pipeline.get_multi_view_condition_input(
                pipeline.condition_embedders["slat_condition_embedder"],
                slat_dicts,
                pipeline.slat_condition_input_mapping,
            )
            cond_args = tuple(cond_args) + (coords.detach().cpu().numpy().astype(np.int32, copy=False),)
            latent_shape = (slat_dicts[0]["image"].shape[0], coords.shape[0], 8)
            prev_steps = slat_gen.inference_steps
            slat_gen.inference_steps = teacher_steps
            with inject_generator_multi_view(
                slat_gen,
                num_views=len(slat_dicts),
                num_steps=teacher_steps,
                mode="multidiffusion",
            ):
                teacher = slat_gen(latent_shape, device_s2, *cond_args, **cond_kwargs)
            slat_gen.inference_steps = prev_steps
    cfg_mod.strength = 0.0
    x1 = teacher[0].detach().float()
    if x1.dim() == 2:
        x1 = x1.unsqueeze(0)

    cond_args_train = []
    for a in cond_args[:-1]:
        cond_args_train.append(a.detach().to(device_s2) if torch.is_tensor(a) else a)
    cond_args_train.append(cond_args[-1])
    cond_args_train = tuple(cond_args_train)
    cond_kwargs_fm = {k: (v.detach() if torch.is_tensor(v) else v) for k, v in cond_kwargs.items()}
    c0 = cond_args_train[0]
    if torch.is_tensor(c0) and c0.shape[0] == len(slat_dicts):
        c_single = c0[0]
        if c_single.dim() == 2:
            c_single = c_single.unsqueeze(0)
        cond_args_fm = (c_single,) + cond_args_train[1:]
        print(f"flow cond: multi-view {tuple(c0.shape)} → view0 {tuple(c_single.shape)}")
    else:
        cond_args_fm = cond_args_train

    ts = float(getattr(slat_gen, "time_scale", 1000.0) or 1000.0)
    STAGE2_PRIOR = Stage2FlowPrior(
        slat_generator=slat_gen,
        cond_args=cond_args_fm,
        cond_kwargs=cond_kwargs_fm,
        coords=coords.detach().cpu(),
        x1_slat=x1.cpu(),
        align=STAGE1_ALIGN,
        time_scale=ts,
        grid_res=64,
    ).to(device_s2)

    with torch.no_grad():
        zt = torch.rand(1, 1, 32, 32, 32, device=device_s2)
        tt = torch.tensor([0.5], device=device_s2)
        vv = STAGE2_PRIOR.velocity(zt, tt)
        # Alignment sanity: densified Stage-2 support vs OCC_GRID_N11 (same NeuS frame)
        dens = STAGE2_PRIOR.slat_feats_to_dense(
            STAGE2_PRIOR.coords_xyz, STAGE2_PRIOR.x1_slat, 64, device_s2, torch.float32
        )
        if OCC_GRID_N11 is not None:
            occ = OCC_GRID_N11.to(device_s2)
            if occ.shape != dens.shape:
                occ = torch.nn.functional.interpolate(
                    occ[None, None], size=dens.shape, mode="trilinear", align_corners=True
                )[0, 0]
            a = (dens > 0.1).float(); b = (occ > 0.5).float()
            inter = (a * b).sum().clamp_min(1e-6)
            union = ((a + b) > 0).float().sum().clamp_min(1e-6)
            iou = float(inter / union)
            # pearson-ish
            dn = (dens - dens.mean()) / dens.std().clamp_min(1e-6)
            on = (occ - occ.mean()) / occ.std().clamp_min(1e-6)
            corr = float((dn * on).mean())
            print(f"  align check vs OCC_GRID_N11: IoU(dens>0.1,occ>0.5)={iou:.3f} corr={corr:.3f}")
            _snap6 = {
                "section": "6.0a",
                "align_iou_vs_occ": float(iou),
                "align_corr_vs_occ": float(corr),
                "n_coords": int(STAGE2_PRIOR.coords_xyz.shape[0]),
                "x1_shape": list(tuple(STAGE2_PRIOR.x1_slat.shape)),
                "USE_EXPORTED_STAGE1_COORDS": bool(globals().get("USE_EXPORTED_STAGE1_COORDS", True)),
                "FLOW_PRIOR_ON_VISIBLE": bool(globals().get("FLOW_PRIOR_ON_VISIBLE", False)),
                "RUN_TS": str(globals().get("RUN_TS")),
            }
            import json as _json
            _p6 = OUTPUT_DIR / "debug_snapshot_6_0a.json"
            _p6.write_text(_json.dumps(_snap6, indent=2))
            print("=== DEBUG SNAPSHOT §6.0a (paste this block) ===")
            print(_json.dumps(_snap6, indent=2))
            print("=== END SNAPSHOT ===")

            if iou < 0.05:
                print("  ⚠ VERY LOW overlap — Sim3/meta mismatch or wrong Stage-1 export. Do not trust this prior.")
            try:
                import matplotlib.pyplot as plt
                mid = dens.shape[0] // 2
                fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
                axs[0].imshow(dens[mid].cpu(), origin="lower"); axs[0].set_title("Stage-2 dens (z mid)")
                axs[1].imshow(occ[mid].cpu(), origin="lower"); axs[1].set_title("OCC NeuS (z mid)")
                axs[2].imshow((dens * occ)[mid].cpu(), origin="lower"); axs[2].set_title("product")
                for ax in axs: ax.axis("off")
                fig.tight_layout()
                _png = OUTPUT_DIR / "stage2_align_check.png"
                fig.savefig(_png, dpi=120, bbox_inches="tight"); plt.show()
                print("  saved", _png)
            except Exception as _e:
                print("  align plot skipped:", _e)
    print(f"✓ STAGE2_PRIOR ready | x1 {tuple(x1.shape)} | velocity {tuple(vv.shape)} | time_scale={ts}")
    print("  densify = coords/64−0.5 → Sim3 → NeuS soft-splat (NOT AABB)")
    print("  (velocity via slat_generator.reverse_fn — same module FlowMatching.loss uses)")
    return STAGE2_PRIOR


if USE_FLOW_PRIOR and str(FLOW_PRIOR_MODE).lower() in ("stage2", "slat", "mvsam3d"):
    try:
        build_stage2_flow_prior()
    except Exception as e:
        STAGE2_PRIOR = None
        print(f"✗ Stage-2 prior failed: {type(e).__name__}: {e}")
        print("  Need same stack as mvsam3d_stage2LoRA (HF_TOKEN + sam-3d-objects). Or FLOW_PRIOR_MODE='occupancy'.")
        raise
else:
    print(f"Skipping Stage-2 load (USE_FLOW_PRIOR={USE_FLOW_PRIOR}, MODE={FLOW_PRIOR_MODE})")

In [ ]:
# 6.0 — Build full model + optimizers (+ optional Stage-1 prefit)
device = torch.device(DEVICE)

enc = HashGridEncoding(
    n_levels=HASH_N_LEVELS,
    n_features_per_level=HASH_N_FEATURES,
    log2_hashmap_size=HASH_LOG2_SIZE if HAS_TCNN else min(HASH_LOG2_SIZE, 18),
    base_resolution=HASH_BASE_RES,
    max_resolution=HASH_MAX_RES,
).to(device)
sdf_net = HashSDF(enc, hidden=SDF_HIDDEN, n_layers=SDF_N_LAYERS, geo_init_bias=0.5).to(device)
bg_net = BackgroundNeRF(hidden=BG_HIDDEN).to(device)
color_net = ColorNet(feat_dim=SDF_HIDDEN, hidden=SDF_HIDDEN).to(device)
poses = LearnablePoses(torch.from_numpy(scene["c2ws"]).to(device).float()).to(device)

MASKS_T = torch.from_numpy(np.stack(scene["masks"]).astype(np.float32))
Ks_T = torch.from_numpy(np.asarray(scene["Ks"], np.float32))

_occ = OCC_GRID_N11.to(device) if OCC_GRID_N11 is not None else None
if FLOW_PRIOR_MODE == "occupancy" and _occ is None:
    print("⚠ occupancy prior requested but no Stage-1 grid — disabling flow prior")
    prior = None
    USE_FLOW_PRIOR_EFF = False
else:
    prior = build_flow_prior(
        FLOW_PRIOR_MODE if USE_FLOW_PRIOR else "off",
        occ_grid_n11=_occ,
        stage2_prior=globals().get("STAGE2_PRIOR"),
    )
    if prior is not None:
        prior = prior.to(device)
    USE_FLOW_PRIOR_EFF = USE_FLOW_PRIOR and prior is not None

opt_sdf = torch.optim.Adam(
    list(sdf_net.parameters()) + list(color_net.parameters()) + list(bg_net.parameters()),
    lr=LR_SDF,
)
opt_pose = torch.optim.Adam(poses.parameters(), lr=LR_POSE)

print(
    f"params SDF+color+bg: "
    f"{sum(p.numel() for p in sdf_net.parameters())+sum(p.numel() for p in color_net.parameters())+sum(p.numel() for p in bg_net.parameters()):,}"
)
print(f"params poses: {sum(p.numel() for p in poses.parameters()):,}")
print(f"hash backend: {enc.backend} | flow prior: {FLOW_PRIOR_MODE if USE_FLOW_PRIOR_EFF else 'off'}")
print(f"Stage-1: {HAS_STAGE1} | hard_mask_route={HARD_MASK_ROUTE}")

if USE_STAGE1_PREFIT and HAS_STAGE1:
    print("=== Stage-1 Hash-SDF prefit ===")
    prefit_hash_sdf(sdf_net, n_iters=SDF_PREFIT_ITERS, batch=SDF_PREFIT_BATCH, lr=SDF_PREFIT_LR)
else:
    print("skipping Stage-1 prefit")


### 6.0c — Flow prior schedule (post–Stage-1-prefit)

**Prefit:** Stage-1 occupancy only (no Stage-2).

**Idea:** photo alone is an ill-behaved surface early — use Stage-2 as a strong
basin-finder / smoother first, then hand off so photo owns fidelity near the min.

**Weight $W_{\mathrm{flow}}$ (hold → cosine):**
- Hold `W_FLOW_START=0.55` for first `FLOW_HOLD_FRAC=35%` of main train
- Then cosine down to `W_FLOW_END=0.06` over the rest of `END_ITER`
- Amortized pull ≈ `W/period`: early ~`0.55/12≈0.046`, late ~`0.06/100≈0.0006`

**Cadence:** adaptive ~12→25→50→100 (dense in hold, sparse late). **Spatial:** vis-gated by default.

In [ ]:
_sched_png = plot_flow_prior_schedule(OUTPUT_DIR / "flow_prior_schedule.png")
if Path("/content/drive/MyDrive").exists():
    import shutil
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(_sched_png, DRIVE_OUT / _sched_png.name)
    print("→ Drive", DRIVE_OUT / _sched_png.name)
for _it in (0, FLOW_ANNEAL_ITERS // 4, FLOW_ANNEAL_ITERS // 2, FLOW_ANNEAL_ITERS, END_ITER - 1):
    _w, _g = flow_prior_schedule(_it)
    print(f"  it={_it:5d}  W_flow={_w:.3f}  gate={_g:.3f}  (vis_weight = gate * V)")


In [ ]:
# 6.1 — Training loop (visibility-gated photo ↔ prior/flow)
from collections import defaultdict
import time
import math
import torch.nn.functional as F

history = defaultdict(list)
# also logs: w_flow, flow_gate (temporal prior schedule)
t0 = time.time()
sdf_net.train(); bg_net.train(); color_net.train(); poses.train()


def bake_vis_grid(c2ws_np, res=32, max_views=VIS_MAX_VIEWS):
    lin = torch.linspace(-1, 1, res, device=device)
    zz, yy, xx = torch.meshgrid(lin, lin, lin, indexing="ij")
    pts = torch.stack([xx, yy, zz], dim=-1).reshape(-1, 3)
    V = multi_view_point_visibility(
        pts, c2ws_np, scene["Ks"], MASKS_T,
        depths=scene.get("depths"),
        max_views=max_views,
    )
    return V.reshape(res, res, res)


vis_grid = None

for it in range(END_ITER):
    gamma = anneal_gamma(it, HASH_N_LEVELS, ANNEAL_ITERS, ANNEAL_START_LVL)
    w_flow, flow_gate = flow_prior_schedule(it)
    # Stage-2 only: override V-gate for flow distill (occ prior still uses flow_gate schedule).
    if not FLOW_PRIOR_ON_VISIBLE:
        flow_vis_gate = 1.0   # weight ∝ (1−V): blind spots only
    else:
        flow_vis_gate = 0.0   # weight = 1 everywhere (incl. visible)
    c2ws = poses()
    c2ws_np = c2ws.detach().cpu().numpy()

    if vis_grid is None or (it % 500 == 0):
        vis_grid = bake_vis_grid(c2ws_np, res=32)

    rays_o, rays_d, rgb_gt, msk, dep, vi = sample_rays(
        scene["images"], scene["masks"], scene["depths"],
        scene["Ks"], c2ws, n_rays=BATCH_RAYS, device=device,
    )
    out = render_dual_field(
        rays_o, rays_d, sdf_net, bg_net, color_net,
        gamma=gamma, n_samples=N_SAMPLES, n_bg=N_BG_SAMPLES,
        mask_rays=msk, hard_mask_route=HARD_MASK_ROUTE,
    )

    err = (out["rgb"] - rgb_gt) ** 2
    w_pix = 0.05 + 0.95 * msk
    L_photo = (err * w_pix[:, None]).sum() / (w_pix.sum() * 3 + 1e-6)

    O = out["O_fg"].clamp(1e-4, 1 - 1e-4)
    L_mask = F.binary_cross_entropy(O, msk.clamp(0, 1))
    L_eik = out["eikonal"]

    L_depth = torch.zeros((), device=device)
    valid_d = (dep > 1e-4) & (msk > 0.5)
    if valid_d.any():
        L_depth = F.l1_loss(out["depth"][valid_d], dep[valid_d])

    # continuous Stage-1 occupancy prior on UNSEEN points
    L_occ = torch.zeros((), device=device)
    if HAS_STAGE1 and W_OCC_PRIOR > 0:
        pts = torch.randn(OCC_PRIOR_N, 3, device=device)
        pts = pts / (pts.norm(dim=-1, keepdim=True) + 1e-6) * torch.rand(OCC_PRIOR_N, 1, device=device) ** (1 / 3)
        with torch.no_grad():
            tgt = pseudo_sdf_from_prob(sample_prob(pts)).unsqueeze(-1)
            V_pts = multi_view_point_visibility(
                pts.detach(), c2ws_np, scene["Ks"], MASKS_T,
                depths=scene.get("depths"), max_views=VIS_MAX_VIEWS,
            )
            # same scaffold→chisel gate as flow: early global prior, late (1-V)
            V_pts_gated = float(flow_gate) * V_pts
        pred, _ = sdf_net(pts, gamma=gamma)
        L_occ = visibility_weighted_point_loss(pred.squeeze(-1), tgt.squeeze(-1), V_pts_gated)

    # Stage-2 distill: adaptive cadence (expensive reverse_fn) + visibility gate
    L_flow = torch.zeros((), device=device)
    flow_period = flow_call_period(it)
    do_flow = (
        USE_FLOW_PRIOR_EFF
        and w_flow > 1e-8
        and (it % flow_period == 0)
    )
    if do_flow:
        fres = flow_proxy_res(it)
        vg = F.interpolate(
            vis_grid[None, None].float(), size=(fres, fres, fres),
            mode="trilinear", align_corners=True,
        )[0, 0]
        # API: err*(1-vis_weight); vis_weight = flow_vis_gate * V
        vis_for_flow = float(flow_vis_gate) * vg
        L_flow, _flow_info = flow_distillation_loss(
            sdf_net, prior, res=fres,
            gamma=gamma, cond=FLOW_COND_PROMPT, vis_weight=vis_for_flow,
            eta_photo=FLOW_ODE_ETA if it > WARMUP_END else 0.0,
        )

    with torch.no_grad():
        surf = ray_expected_points(rays_o, rays_d, out["weights_fg"].detach(), out["z"].detach())
        V_ray = multi_view_point_visibility(
            surf, c2ws_np, scene["Ks"], MASKS_T,
            depths=scene.get("depths"), max_views=VIS_MAX_VIEWS,
        )
        V_ray = torch.where(out["O_fg"].detach() > 0.05, V_ray, torch.zeros_like(V_ray))
    L_unseen = L_occ if (HAS_STAGE1 and W_OCC_PRIOR > 0) else L_flow
    L_gated, v_used = visibility_gate_losses(
        L_photo, L_unseen if float(L_unseen) != 0 else L_photo * 0, V_ray.mean()
    )

    loss = (
        W_PHOTO * L_gated
        + W_MASK * L_mask
        + W_EIKONAL * L_eik
        + W_DEPTH * L_depth
        + W_OCC_PRIOR * L_occ
        + (w_flow * L_flow if do_flow else 0.0)
    )

    opt_sdf.zero_grad(set_to_none=True)
    opt_pose.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(sdf_net.parameters()) + list(color_net.parameters()) + list(bg_net.parameters()), 5.0
    )
    opt_sdf.step()
    if it >= POSE_START_ITER:
        opt_pose.step()

    if it < WARMUP_END:
        sc = 0.1 + 0.9 * it / WARMUP_END
    else:
        sc = 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (it - WARMUP_END) / max(1, END_ITER - WARMUP_END)))
    for pg in opt_sdf.param_groups:
        pg["lr"] = LR_SDF * sc
    for pg in opt_pose.param_groups:
        pg["lr"] = LR_POSE * sc

    history["loss"].append(float(loss))
    history["photo"].append(float(L_photo))
    history["mask"].append(float(L_mask))
    history["eik"].append(float(L_eik))
    history["occ"].append(float(L_occ))
    history["flow"].append(float(L_flow) if torch.is_tensor(L_flow) else 0.0)
    history["V"].append(float(v_used))
    history["gamma"].append(gamma)
    history["w_flow"].append(float(w_flow))
    history["flow_gate"].append(float(flow_gate))
    history.setdefault("flow_vis_gate", []).append(float(flow_vis_gate))

    if it % 100 == 0 or it == END_ITER - 1:
        dt = time.time() - t0
        print(
            f"iter {it:6d}/{END_ITER}  loss={float(loss):.4f}  photo={float(L_photo):.4f}  "
            f"mask={float(L_mask):.4f}  eik={float(L_eik):.4f}  occ={float(L_occ):.4f}  "
            f"flow={float(L_flow):.4f}  V={float(v_used):.3f}  "
            f"Wflow={w_flow:.3f}  gate={flow_gate:.2f}  flowVis={flow_vis_gate:.0f}  fEvery={flow_period}  γ={gamma:.2f}  [{dt:.0f}s]"
        )

ckpt = {
    "sdf": sdf_net.state_dict(),
    "bg": bg_net.state_dict(),
    "color": color_net.state_dict(),
    "poses": poses.state_dict(),
    "history": dict(history),
    "config": {
        "CASE_NAME": CASE_NAME, "END_ITER": END_ITER, "HASH_N_LEVELS": HASH_N_LEVELS,
        "ANNEAL_ITERS": ANNEAL_ITERS, "FLOW_ANNEAL_ITERS": FLOW_ANNEAL_ITERS,
        "W_FLOW_START": W_FLOW_START, "W_FLOW_END": W_FLOW_END, "FLOW_HOLD_FRAC": FLOW_HOLD_FRAC,
        "PROXY_RES": PROXY_RES,
        "FLOW_PRIOR_MODE": FLOW_PRIOR_MODE, "FLOW_PRIOR_ON_VISIBLE": FLOW_PRIOR_ON_VISIBLE,
        "USE_STAGE1_PREFIT": USE_STAGE1_PREFIT,
    },
}
ckpt_path = OUTPUT_DIR / "gfhashba_ckpt.pth"
torch.save(ckpt, ckpt_path)
print("Saved", ckpt_path)

if Path("/content/drive/MyDrive").exists():
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy2(ckpt_path, DRIVE_OUT / ckpt_path.name)
    print("→ Drive", DRIVE_OUT)


In [ ]:
# 6.2 — Loss curves (+ flow schedule traces)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
keys = ["loss", "photo", "mask", "eik", "occ", "flow", "V", "gamma", "w_flow", "flow_gate"]
for ax, k in zip(axes.ravel(), keys):
    y = history.get(k, [])
    ax.plot(y, lw=1)
    ax.set_title(k)
    ax.grid(alpha=0.3)
    if k not in ("V", "gamma", "w_flow", "flow_gate") and y and min(y) > 0:
        ax.set_yscale("log")
# hide unused if any
for ax in axes.ravel()[len(keys):]:
    ax.axis("off")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "train_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("γ final:", history["gamma"][-1] if history.get("gamma") else None)
print("W_flow final:", history["w_flow"][-1] if history.get("w_flow") else None,
      "| gate final:", history["flow_gate"][-1] if history.get("flow_gate") else None)


## 7 — Extract mesh (marching cubes on Hash-SDF)

Evaluate SDF on a dense grid → marching cubes.

**Health gates (new):** if `sdf range` is almost all-positive (e.g. `[-0.06, 0.33]`) or the mesh
has dozens of connected components, you get floating slabs — **stop and inspect**, do not trust the GLB.

In [ ]:
# 7.0 — Marching cubes export + mesh health gates
import sys
from pathlib import Path
import numpy as np
import torch
import trimesh
import matplotlib.pyplot as plt

_repo_candidates = [
    Path("/content/GSV-MSV"), Path("/content"), Path.cwd(),
    Path("/Users/taiganishida/Documents/GSV-MSV"),
]
for _r in _repo_candidates:
    if (_r / "gfhashba_align.py").exists() and str(_r) not in sys.path:
        sys.path.insert(0, str(_r))
try:
    from gfhashba_align import mesh_health, occ_bbox_from_grid, sdf_volume_health
    _HAS_ALIGN_LIB = True
except Exception as _e:
    print("⚠ gfhashba_align missing — inline health fallbacks:", _e)
    _HAS_ALIGN_LIB = False

try:
    from skimage.measure import marching_cubes
except ImportError:
    import subprocess, sys as _sys
    subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q", "scikit-image"])
    from skimage.measure import marching_cubes

sdf_net.eval()
RES = 256
gamma_final = float(HASH_N_LEVELS)

lin = torch.linspace(-1, 1, RES, device=device)
zz, yy, xx = torch.meshgrid(lin, lin, lin, indexing="ij")
pts = torch.stack([xx, yy, zz], dim=-1).reshape(-1, 3)

vols = []
with torch.no_grad():
    chunk = 65536
    for i in range(0, pts.shape[0], chunk):
        s, _ = sdf_net(pts[i:i + chunk], gamma=gamma_final)
        vols.append(s.squeeze(-1).float().cpu())
vol = torch.cat(vols).reshape(RES, RES, RES).numpy()

# ---- SDF health (catches floating-slab mode) ----
if _HAS_ALIGN_LIB:
    _sh = sdf_volume_health(vol, 0.0)
else:
    _fi = float((vol < 0).mean())
    _sh = {
        "vmin": float(vol.min()), "vmax": float(vol.max()), "crosses_zero": bool(vol.min() < 0 < vol.max()),
        "frac_inside": _fi, "ok_sign_balance": (0.01 <= _fi <= 0.85) and (vol.min() < 0 < vol.max()),
        "ok_range": float(vol.max() - vol.min()) > 1e-3,
    }
print(
    f"SDF health: range=[{_sh['vmin']:.3f},{_sh['vmax']:.3f}]  "
    f"frac_inside={_sh['frac_inside']:.4f}  crosses0={_sh['crosses_zero']}  "
    f"ok_sign={_sh['ok_sign_balance']}"
)

# mid-slice SDF preview
_mid = RES // 2
fig, axs = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, sl, t in (
    (axs[0], vol[_mid], "SDF z mid"),
    (axs[1], vol[:, _mid, :], "SDF y mid"),
    (axs[2], (vol < 0).astype(np.float32)[_mid], "inside (sdf<0) z mid"),
):
    ax.imshow(sl, origin="lower", cmap="coolwarm" if "SDF" in t else "gray")
    ax.set_title(t); ax.axis("off")
fig.tight_layout()
_sdf_png = OUTPUT_DIR / "sdf_slices_pre_mc.png"
fig.savefig(_sdf_png, dpi=120, bbox_inches="tight")
plt.show()
print("saved", _sdf_png)

if not _sh["ok_sign_balance"]:
    print(
        "⚠ SDF sign balance BAD — marching cubes at 0 will produce floating debris "
        "(your previous [-0.06,0.33]-style failure). Prefer retrain / stronger prefit; "
        "trying adaptive level as last resort."
    )
    # pick a level near a low percentile of |grad|-weighted distribution: use median of negative if any, else 20th pct
    if (vol < 0).any():
        level = float(np.median(vol[vol < 0]))
    else:
        level = float(np.percentile(vol, 20))
    print(f"  adaptive MC level={level:.4f}")
else:
    level = 0.0

try:
    verts, faces, normals, _ = marching_cubes(vol, level=level)
except Exception as e:
    print("MC failed:", e, "— retry median")
    level = float(np.median(vol))
    verts, faces, normals, _ = marching_cubes(vol, level=level)

verts = verts / (RES - 1) * 2.0 - 1.0
mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_normals=normals, process=False)
print(f"mesh: {len(mesh.vertices)} verts, {len(mesh.faces)} faces | MC level={level:.4f}")

# ---- mesh health ----
_occ_bb = None
if OCC_GRID_N11 is not None:
    _occ_np = OCC_GRID_N11.detach().float().cpu().numpy()
    if _HAS_ALIGN_LIB:
        _occ_bb = occ_bbox_from_grid(_occ_np, 0.5)
if _HAS_ALIGN_LIB:
    _mh = mesh_health(np.asarray(mesh.vertices), np.asarray(mesh.faces), occ_bbox=_occ_bb)
else:
    _mh = {"n_verts": len(mesh.vertices), "n_faces": len(mesh.faces), "n_components": -1,
           "largest_frac": float("nan"), "ok": len(mesh.faces) >= 500, "reason": "no align lib"}
print(
    f"Mesh health: comps={_mh.get('n_components')}  largest_frac={_mh.get('largest_frac', float('nan')):.3f}  "
    f"bbox_iou_vs_occ={_mh.get('bbox_iou_vs_occ', float('nan'))}  ok={_mh.get('ok')}  ({_mh.get('reason')})"
)
if not _mh.get("ok", False):
    print("⚠ MESH HEALTH FAIL — floating debris / empty / misaligned. Do not trust this GLB for A/B.")

# optional: map back to world via NeuS scale_mat
scale_mat = np.asarray(scene.get("scale_mat", np.eye(4)), np.float64)
ones = np.ones((len(verts), 1))
verts_h = np.concatenate([verts, ones], 1)
verts_w = (scale_mat @ verts_h.T).T[:, :3]
mesh_w = trimesh.Trimesh(vertices=verts_w, faces=faces, process=False)

out_norm = OUTPUT_DIR / "gfhashba_mesh_norm.glb"
out_world = OUTPUT_DIR / "gfhashba_mesh_world.glb"
mesh.export(out_norm); mesh_w.export(out_world)
mesh.export(OUTPUT_DIR / "gfhashba_mesh_norm.ply")
print("saved", out_norm, "and", out_world)

fig = plt.figure(figsize=(12, 4))
for j, (az, el) in enumerate([(-60, 20), (40, 25), (90, 75)]):
    ax = fig.add_subplot(1, 3, j + 1, projection="3d")
    ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2], triangles=faces,
                    cmap="viridis", linewidth=0, antialiased=False)
    ax.view_init(elev=el, azim=az); ax.set_box_aspect((1, 1, 1))
    ax.set_title(f"GF-HashBA ({az},{el})")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "mesh_preview.png", dpi=120, bbox_inches="tight")
plt.show()


# ---- paste-friendly summary for chat debug ----
_snap7 = {
    "section": "7",
    "sdf_vmin": float(_sh["vmin"]),
    "sdf_vmax": float(_sh["vmax"]),
    "frac_inside": float(_sh["frac_inside"]),
    "ok_sign_balance": bool(_sh["ok_sign_balance"]),
    "mc_level": float(level),
    "n_verts": int(len(mesh.vertices)),
    "n_faces": int(len(mesh.faces)),
    "n_components": _mh.get("n_components"),
    "largest_frac": _mh.get("largest_frac"),
    "bbox_iou_vs_occ": _mh.get("bbox_iou_vs_occ"),
    "mesh_ok": bool(_mh.get("ok", False)),
    "mesh_reason": str(_mh.get("reason")),
    "FLOW_PRIOR_ON_VISIBLE": bool(globals().get("FLOW_PRIOR_ON_VISIBLE", False)),
    "RUN_TS": str(globals().get("RUN_TS")),
    "DRIVE_OUT": str(globals().get("DRIVE_OUT")),
}
import json as _json
_snap7_path = OUTPUT_DIR / "debug_snapshot_7.json"
_snap7_path.write_text(_json.dumps(_snap7, indent=2))
print("=== DEBUG SNAPSHOT §7 (paste this block) ===")
print(_json.dumps(_snap7, indent=2))
print("=== END SNAPSHOT ===")

if Path("/content/drive/MyDrive").exists():
    import shutil
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    for p in (out_norm, out_world, OUTPUT_DIR / "gfhashba_mesh_norm.ply",
              OUTPUT_DIR / "mesh_preview.png", OUTPUT_DIR / "train_curves.png",
              OUTPUT_DIR / "flow_prior_schedule.png", OUTPUT_DIR / "align_diag_s1.png",
              OUTPUT_DIR / "sdf_slices_pre_mc.png"):
        if Path(p).exists():
            shutil.copy2(p, DRIVE_OUT / Path(p).name)
    print("→ Drive", DRIVE_OUT)

## 8 — Stage-2 / Trellis velocity

**Default now:** `FLOW_PRIOR_MODE="stage2"` via §6.0a — real MV-SAM3D `slat_generator.reverse_fn`
velocity, **Sim3-mapped** into the NeuS grid (same frame as Stage-1 prefit; exported coords + soft-splat).
Check `stage2_align_check.png` after §6.0a — IoU vs `OCC_GRID_N11` should not be near zero.

Optional LoRA: set `STAGE2_LORA_DIR` to a saved `lora_adapter/` from `mvsam3d_stage2LoRA.ipynb`.

**Visible toggle:** `FLOW_PRIOR_ON_VISIBLE=False` → Stage-2 only on blind spots `(1−V)`; `True` → also on visible.

Fallback modes:
- `"occupancy"` — Stage-1 64³ teacher only (old stand-in)
- `"external"` — `prior.attach(fn)` for a custom Trellis velocity
- `"off"` — no generative distill


## 9 — Relation to the BA-NeuS notebooks

| | `sam_baneus_ultrra.ipynb` | **GF-HashBA (this)** |
|--|--|--|
| Geometry | MLP NeuS SDF | **Hash-SDF** + bg NeRF |
| Topology prior | Stage-1 occupancy MSE | Same + **flow distillation** on unseen voxels |
| Visibility | (none / prior everywhere) | **Multi-view V(x)** gates photo ↔ prior |
| Early BA stability | MLP spectral bias | **Hash annealing** $W_l(\gamma)$ |

**Need from sam_baneus:** `cameras_sphere.npz`, `baneus_scene_meta.npz`, `stage1/stage1_occupancy.npz`.
